# 450 — Secondary Genomic Context Characterization

## Objective

Characterize somatic mutation context associated with the three frozen Phase 4
cross-system consensus transcriptomic programs in the TCGA primary-tumor cohort.

The analysis integrates the frozen TCGA somatic-mutation resource audited in
notebook 108 with the frozen consensus tumor scores produced in notebook 401.

The objective is to identify recurrent gene-level mutation contexts associated
with consensus-program variation while preserving lineage structure and
explicitly distinguishing recurrent association from gene-specific mechanism.

## Frozen upstream inputs

Notebook 450 operates on frozen upstream objects:

- the TCGA primary-tumor multi-omic cohort and sample mapping;
- the audited exact-sample somatic-mutation handoff from notebook 108;
- the three frozen consensus transcriptomic programs and tumor scores from
  notebook 401; and
- frozen tumor-side confounder covariates where available.

Consensus-program identities, orientations, gene weights, scores, and upstream
eligibility are not modified on the basis of genomic-context results.

## Analytical status

Notebook 450 is a secondary molecular-characterization analysis over a frozen
program universe.

Somatic mutation associations are evaluated as computational associations.
They do not establish causal mechanisms, driver status, therapeutic relevance,
clinical prediction, or absolute wild-type status.

Negative, heterogeneous, lineage-restricted, or non-recoverable results are
valid scientific outcomes.

## Mutation representation and eligibility boundary

Exact frozen tumor-sample MAFs define the mutation-resource cohort.

When multiple exact-sample MAFs are available for a case, qualifying variants
are represented by their union followed by deterministic genomic-key
deduplication; no arbitrary file selection is introduced.

Structurally empty MAFs are not interpreted as mutation-free observations.
Cases for which all retained MAFs are structurally empty remain indeterminate
and are excluded from primary binary gene-state inference.

Primary gene state therefore distinguishes:

- `qualifying_somatic_variant_observed`; and
- `no_qualifying_somatic_variant_observed`

within informative mutation-resource cases. The latter is not interpreted as
absolute wild type.

Gene and project eligibility rules are frozen from mutation prevalence and
sample support before program-association results are inspected.

## Lineage-aware analytical boundary

Cancer lineage is treated as a required structural component of the analysis.

Primary gene × program inference uses project-adjusted models restricted to
projects with prespecified mutated and non-mutated sample support. Within-project
effects and leave-one-project-out analyses characterize directional consistency
and lineage dependence.

A pooled association is not sufficient to establish cross-cancer recurrence.
The recurrent-association designation additionally requires prespecified
cross-project directional support and leave-one-project-out stability.

Lineage-aware recurrence describes reproducibility of an association across
eligible TCGA projects; it does not establish biological universality across
cancer types.

## Confounding, multiplicity, and sensitivity analyses

The primary model adjusts for TCGA project without introducing covariates whose
coverage or provenance would materially alter the frozen analysis cohort.

Prespecified sensitivity analyses evaluate:

- tumor purity;
- proliferation;
- restriction to single-MAF cases; and
- inclusion of `Splice_Region` in the qualifying-variant definition.

Sensitivity analyses characterize stability of primary associations and cannot
rescue associations that fail the primary inferential framework.

Multiplicity is controlled across the complete frozen primary gene × program
hypothesis family using Benjamini–Hochberg FDR.

Observed qualifying-variant burden is evaluated separately as an exploratory
confounding diagnostic. Because callable territory is not available, this
quantity is not interpreted as tumor mutational burden (TMB). Conditioning on
background observed variant burden does not redefine the primary model,
significance family, recurrence criterion, or sensitivity-stability criterion.

## Scope and methodological boundary

Notebook 450 does not:

- redefine or reoptimize consensus programs;
- infer causal relationships from mutation–program associations;
- interpret absence of a qualifying variant as definitive wild type;
- infer driver status from recurrence or statistical significance;
- introduce copy-number alteration analysis;
- construct TMB without a defensible callable-territory denominator;
- use post hoc burden stability to redefine primary significance or recurrence;
- promote individual genes as validated targets; or
- use downstream functional, pharmacogenomic, or perturbational evidence to
  select genomic-context associations.

The resulting gene-level associations constitute secondary genomic context for
the frozen consensus programs.

## Expected output

Notebook 450 publishes one downstream-consumable characterization artifact under
`data/processed/secondary_characterization/`:

- `450_primary_gene_program_associations.csv` — complete frozen primary
  gene × consensus-program association table, including primary effect estimates,
  multiple-testing results, lineage-aware recurrence characterization,
  prespecified sensitivity stability, and the exploratory focal-excluded
  background observed-variant-burden diagnostic.

Lineage-specific estimates, leave-one-project-out fits, intermediate mutation
tables, sensitivity-specific result tables, overlap analyses, and descriptive
summaries remain reproducible within the notebook and are not persisted as
separate downstream artifacts.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json

import pandas as pd

from pancancer_epigenetics.utils.paths import Paths, project_relative_path

In [2]:
# =============================================================================
# Resolve frozen notebook-108 artifact paths
# =============================================================================

MUTATION_ARTIFACT_IDS = (
    "phase1.108.primary_file_handoff",
    "phase1.108.case_eligibility",
    "phase1.108.download_validation",
    "phase1.108.case_payload_status",
)

with Paths.artifact_registry.open("r", encoding="utf-8") as handle:
    artifact_registry = json.load(handle)

mutation_artifact_paths = {
    artifact_id: Paths.root / artifact_registry["artifacts"][artifact_id]["path"]
    for artifact_id in MUTATION_ARTIFACT_IDS
}

for artifact_id, path in mutation_artifact_paths.items():
    print(f"{artifact_id}: {project_relative_path(path)}")

phase1.108.primary_file_handoff: data/interim/genomics/108_tcga_somatic_mutation_primary_file_handoff.csv
phase1.108.case_eligibility: data/interim/metadata/108_tcga_somatic_mutation_case_eligibility.csv
phase1.108.download_validation: data/interim/genomics/108_tcga_somatic_mutation_download_validation.csv
phase1.108.case_payload_status: data/interim/metadata/108_tcga_somatic_mutation_case_payload_status.csv


In [3]:
# =============================================================================
# Validate frozen notebook-108 input contract
# =============================================================================

mutation_input_contract = []

for artifact_id, path in mutation_artifact_paths.items():
    entry = artifact_registry["artifacts"][artifact_id]

    mutation_input_contract.append(
        {
            "artifact_id": artifact_id,
            "status": entry["status"],
            "exists": path.is_file(),
        }
    )

mutation_input_contract = pd.DataFrame(mutation_input_contract)

print(
    "All inputs frozen:",
    bool(mutation_input_contract["status"].eq("frozen").all()),
)
print(
    "All input files present:",
    bool(mutation_input_contract["exists"].all()),
)

mutation_input_contract

All inputs frozen: True
All input files present: True


,artifact_id,status,exists
0,phase1.108.primary_file_handoff,frozen,True
1,phase1.108.case_eligibility,frozen,True
2,phase1.108.download_validation,frozen,True
3,phase1.108.case_payload_status,frozen,True


In [4]:
# =============================================================================
# Load frozen notebook-108 mutation handoff
# =============================================================================

primary_file_handoff = pd.read_csv(
    mutation_artifact_paths["phase1.108.primary_file_handoff"],
    dtype="string",
)

case_eligibility = pd.read_csv(
    mutation_artifact_paths["phase1.108.case_eligibility"],
    dtype="string",
)

download_validation = pd.read_csv(
    mutation_artifact_paths["phase1.108.download_validation"],
)

case_payload_status = pd.read_csv(
    mutation_artifact_paths["phase1.108.case_payload_status"],
)

print("Primary file handoff:", primary_file_handoff.shape)
print("Case eligibility:", case_eligibility.shape)
print("Download validation:", download_validation.shape)
print("Case payload status:", case_payload_status.shape)

Primary file handoff: (9426, 16)
Case eligibility: (9965, 7)
Download validation: (9426, 10)
Case payload status: (9192, 7)


In [5]:
# =============================================================================
# Inspect frozen mutation-handoff fields
# =============================================================================

for name, frame in {
    "primary_file_handoff": primary_file_handoff,
    "case_eligibility": case_eligibility,
    "download_validation": download_validation,
    "case_payload_status": case_payload_status,
}.items():
    print(f"\n{name}")
    print(frame.columns.tolist())


primary_file_handoff
['file_id', 'gdc_case_id', 'tcga_case_barcode', 'project_id', 'frozen_tumor_sample_barcode', 'tumor_sample_barcode', 'tumor_aliquot_id', 'tumor_aliquot_barcode', 'normal_aliquot_id', 'normal_aliquot_barcode', 'tumor_portion_barcode', 'tumor_analyte', 'file_name', 'file_md5', 'file_size', 'n_mafs_for_frozen_sample']

case_eligibility
['tcga_case_barcode', 'frozen_tumor_sample_barcode', 'project_id', 'has_mutation_case', 'n_exact_sample_mafs', 'mutation_resource_status', 'mutation_resource_eligible']

download_validation
['id', 'filename', 'md5', 'size', 'observed_size', 'exists', 'size_matches', 'observed_md5', 'md5_matches', 'has_variant_row']

case_payload_status
['tcga_case_barcode', 'project_id', 'n_mafs', 'n_mafs_with_variant_rows', 'n_empty_mafs', 'all_mafs_empty', 'mixed_empty_nonempty']


## Outcome-blind mutation-content characterization

Notebook 108 freezes mutation-resource eligibility, exact-sample file identity,
payload validity, and structural empty-MAF status. Here we inspect only the
mutation-content structure required to define a deterministic case-level
representation before loading or examining consensus program scores.

Upstream eligibility is not reconstructed or modified. Detailed
mutation-representation rules are preserved in
`docs/contracts/phase4b/PHASE4B_450_ANALYSIS_CONTRACT.md`.

In [6]:
# =============================================================================
# Resolve frozen MAF payload paths
# =============================================================================

MUTATION_DOWNLOAD_DIR = Paths.tcga / "somatic_mutation"

maf_file_table = primary_file_handoff[
    [
        "file_id",
        "tcga_case_barcode",
        "project_id",
        "frozen_tumor_sample_barcode",
        "file_name",
        "n_mafs_for_frozen_sample",
    ]
].copy()

maf_file_table["maf_path"] = [
    MUTATION_DOWNLOAD_DIR / file_id / file_name
    for file_id, file_name in zip(
        maf_file_table["file_id"],
        maf_file_table["file_name"],
    )
]

missing_maf_files = ~maf_file_table["maf_path"].map(lambda path: path.is_file())

print(f"Resolved MAF files: {len(maf_file_table):,}")
print(f"Represented cases: {maf_file_table['tcga_case_barcode'].nunique():,}")
print(f"Missing local MAF files: {int(missing_maf_files.sum()):,}")

if missing_maf_files.any():
    raise FileNotFoundError(
        "One or more frozen MAF payloads are unavailable locally."
    )

Resolved MAF files: 9,426
Represented cases: 9,192
Missing local MAF files: 0


In [7]:
# =============================================================================
# Inspect canonical MAF fields required by notebook 450
# =============================================================================

example_maf_path = maf_file_table["maf_path"].iloc[0]

example_maf_header = pd.read_csv(
    example_maf_path,
    sep="\t",
    comment="#",
    nrows=0,
)

candidate_maf_fields = [
    "Hugo_Symbol",
    "NCBI_Build",
    "Chromosome",
    "Start_Position",
    "End_Position",
    "Variant_Classification",
    "Variant_Type",
    "Reference_Allele",
    "Tumor_Seq_Allele1",
    "Tumor_Seq_Allele2",
    "Tumor_Sample_Barcode",
]

print(f"MAF schema columns: {len(example_maf_header.columns):,}")
print("\nCandidate fields:")
for column in candidate_maf_fields:
    print(f"{column}: {column in example_maf_header.columns}")

MAF schema columns: 140

Candidate fields:
Hugo_Symbol: True
NCBI_Build: True
Chromosome: True
Start_Position: True
End_Position: True
Variant_Classification: True
Variant_Type: True
Reference_Allele: True
Tumor_Seq_Allele1: True
Tumor_Seq_Allele2: True
Tumor_Sample_Barcode: True


In [8]:
# =============================================================================
# Audit mutation annotation and allele representation
# =============================================================================

TECHNICAL_MAF_FIELDS = [
    "NCBI_Build",
    "Variant_Classification",
    "Variant_Type",
    "Reference_Allele",
    "Tumor_Seq_Allele1",
    "Tumor_Seq_Allele2",
]

build_counts = {}
classification_counts = {}
variant_type_counts = {}

allele_pattern_counts = {
    "allele1_ref_allele2_alt": 0,
    "allele1_alt_allele2_ref": 0,
    "both_ref": 0,
    "both_alt_same": 0,
    "both_alt_different": 0,
    "missing_allele_value": 0,
}

total_variant_rows = 0

for maf_path in maf_file_table["maf_path"]:
    maf = pd.read_csv(
        maf_path,
        sep="\t",
        comment="#",
        usecols=TECHNICAL_MAF_FIELDS,
        dtype="string",
        low_memory=False,
    )

    if maf.empty:
        continue

    total_variant_rows += len(maf)

    for value, count in maf["NCBI_Build"].fillna("<NA>").value_counts().items():
        build_counts[value] = build_counts.get(value, 0) + int(count)

    for value, count in (
        maf["Variant_Classification"].fillna("<NA>").value_counts().items()
    ):
        classification_counts[value] = (
            classification_counts.get(value, 0) + int(count)
        )

    for value, count in maf["Variant_Type"].fillna("<NA>").value_counts().items():
        variant_type_counts[value] = (
            variant_type_counts.get(value, 0) + int(count)
        )

    ref = maf["Reference_Allele"]
    allele1 = maf["Tumor_Seq_Allele1"]
    allele2 = maf["Tumor_Seq_Allele2"]

    missing = ref.isna() | allele1.isna() | allele2.isna()

    allele1_is_ref = allele1.eq(ref)
    allele2_is_ref = allele2.eq(ref)
    alleles_equal = allele1.eq(allele2)

    allele_pattern_counts["missing_allele_value"] += int(missing.sum())
    allele_pattern_counts["allele1_ref_allele2_alt"] += int(
        (~missing & allele1_is_ref & ~allele2_is_ref).sum()
    )
    allele_pattern_counts["allele1_alt_allele2_ref"] += int(
        (~missing & ~allele1_is_ref & allele2_is_ref).sum()
    )
    allele_pattern_counts["both_ref"] += int(
        (~missing & allele1_is_ref & allele2_is_ref).sum()
    )
    allele_pattern_counts["both_alt_same"] += int(
        (
            ~missing
            & ~allele1_is_ref
            & ~allele2_is_ref
            & alleles_equal
        ).sum()
    )
    allele_pattern_counts["both_alt_different"] += int(
        (
            ~missing
            & ~allele1_is_ref
            & ~allele2_is_ref
            & ~alleles_equal
        ).sum()
    )

print(f"Variant rows inspected: {total_variant_rows:,}")

print("\nNCBI_Build:")
print(pd.Series(build_counts, dtype="int64").sort_values(ascending=False))

print("\nVariant_Classification:")
print(
    pd.Series(classification_counts, dtype="int64")
    .sort_values(ascending=False)
)

print("\nVariant_Type:")
print(
    pd.Series(variant_type_counts, dtype="int64")
    .sort_values(ascending=False)
)

print("\nTumor-allele representation:")
print(pd.Series(allele_pattern_counts, dtype="int64"))

Variant rows inspected: 2,162,244

NCBI_Build:
GRCh38    2162244
dtype: int64

Variant_Classification:
Missense_Mutation         1341779
Silent                     488495
Nonsense_Mutation          112118
Frame_Shift_Del             85156
Splice_Site                 31208
Frame_Shift_Ins             26050
Intron                      20942
Splice_Region               13689
RNA                         13259
In_Frame_Del                 7684
3'UTR                        7349
5'Flank                      3672
5'UTR                        3554
3'Flank                      3236
Translation_Start_Site       1756
Nonstop_Mutation             1570
In_Frame_Ins                  697
IGR                            30
dtype: int64

Variant_Type:
SNP    2035569
DEL      98549
INS      28054
ONP         63
TNP          9
dtype: int64

Tumor-allele representation:
allele1_ref_allele2_alt    2162244
allele1_alt_allele2_ref          0
both_ref                         0
both_alt_same                    0

In [9]:
# =============================================================================
# Audit candidate genomic variant-key integrity
# =============================================================================

VARIANT_KEY_FIELDS = [
    "NCBI_Build",
    "Chromosome",
    "Start_Position",
    "End_Position",
    "Reference_Allele",
    "Tumor_Seq_Allele2",
]

VARIANT_AUDIT_FIELDS = VARIANT_KEY_FIELDS + [
    "Hugo_Symbol",
    "Variant_Classification",
]

key_missing_counts = {
    field: 0
    for field in VARIANT_KEY_FIELDS
}

invalid_start_positions = 0
invalid_end_positions = 0
start_after_end = 0

files_with_duplicate_keys = 0
duplicate_key_rows = 0
unique_duplicate_keys = 0

for maf_path in maf_file_table["maf_path"]:
    maf = pd.read_csv(
        maf_path,
        sep="\t",
        comment="#",
        usecols=VARIANT_AUDIT_FIELDS,
        dtype="string",
        low_memory=False,
    )

    if maf.empty:
        continue

    for field in VARIANT_KEY_FIELDS:
        key_missing_counts[field] += int(maf[field].isna().sum())

    start = pd.to_numeric(
        maf["Start_Position"],
        errors="coerce",
    )
    end = pd.to_numeric(
        maf["End_Position"],
        errors="coerce",
    )

    invalid_start_positions += int(start.isna().sum())
    invalid_end_positions += int(end.isna().sum())
    start_after_end += int(
        (start.notna() & end.notna() & start.gt(end)).sum()
    )

    duplicated = maf.duplicated(
        subset=VARIANT_KEY_FIELDS,
        keep=False,
    )

    if duplicated.any():
        duplicate_subset = maf.loc[
            duplicated,
            VARIANT_KEY_FIELDS,
        ]

        files_with_duplicate_keys += 1
        duplicate_key_rows += int(duplicated.sum())
        unique_duplicate_keys += len(
            duplicate_subset.drop_duplicates()
        )

print("Candidate key missing values:")
print(pd.Series(key_missing_counts, dtype="int64"))

print(f"\nInvalid start positions: {invalid_start_positions:,}")
print(f"Invalid end positions: {invalid_end_positions:,}")
print(f"Rows with start > end: {start_after_end:,}")

print(f"\nFiles with within-MAF duplicate keys: {files_with_duplicate_keys:,}")
print(f"Rows participating in duplicate keys: {duplicate_key_rows:,}")
print(f"Unique duplicated keys within files: {unique_duplicate_keys:,}")

Candidate key missing values:
NCBI_Build           0
Chromosome           0
Start_Position       0
End_Position         0
Reference_Allele     0
Tumor_Seq_Allele2    0
dtype: int64

Invalid start positions: 0
Invalid end positions: 0
Rows with start > end: 0

Files with within-MAF duplicate keys: 0
Rows participating in duplicate keys: 0
Unique duplicated keys within files: 0


In [10]:
# =============================================================================
# Audit cross-file overlap in multi-MAF cases
# =============================================================================

multi_maf_cases = (
    maf_file_table.groupby("tcga_case_barcode", sort=False)
    .filter(lambda group: len(group) > 1)
)

multi_maf_audit_rows = []

for case_barcode, case_files in multi_maf_cases.groupby(
    "tcga_case_barcode",
    sort=False,
):
    case_frames = []

    for row in case_files.itertuples(index=False):
        maf = pd.read_csv(
            row.maf_path,
            sep="\t",
            comment="#",
            usecols=VARIANT_AUDIT_FIELDS,
            dtype="string",
            low_memory=False,
        )

        if maf.empty:
            continue

        maf = maf.copy()
        maf["_file_id"] = row.file_id
        case_frames.append(maf)

    if not case_frames:
        multi_maf_audit_rows.append(
            {
                "tcga_case_barcode": case_barcode,
                "n_files": len(case_files),
                "total_variant_rows": 0,
                "unique_variant_keys": 0,
                "shared_variant_keys": 0,
                "keys_present_in_all_files": 0,
                "gene_annotation_conflicts": 0,
                "classification_annotation_conflicts": 0,
            }
        )
        continue

    combined = pd.concat(
        case_frames,
        ignore_index=True,
    )

    key_file_counts = (
        combined.groupby(VARIANT_KEY_FIELDS)["_file_id"]
        .nunique()
    )

    annotation_summary = combined.groupby(
        VARIANT_KEY_FIELDS
    ).agg(
        n_genes=("Hugo_Symbol", "nunique"),
        n_classifications=("Variant_Classification", "nunique"),
    )

    multi_maf_audit_rows.append(
        {
            "tcga_case_barcode": case_barcode,
            "n_files": len(case_files),
            "total_variant_rows": len(combined),
            "unique_variant_keys": len(key_file_counts),
            "shared_variant_keys": int((key_file_counts > 1).sum()),
            "keys_present_in_all_files": int(
                (key_file_counts == len(case_files)).sum()
            ),
            "gene_annotation_conflicts": int(
                (annotation_summary["n_genes"] > 1).sum()
            ),
            "classification_annotation_conflicts": int(
                (annotation_summary["n_classifications"] > 1).sum()
            ),
        }
    )

multi_maf_audit = pd.DataFrame(multi_maf_audit_rows)

print(f"Multi-MAF cases audited: {len(multi_maf_audit):,}")

print("\nNumber of retained MAFs per case:")
print(
    multi_maf_audit["n_files"]
    .value_counts()
    .sort_index()
)

print("\nCases with at least one shared genomic variant:")
print(
    int(
        multi_maf_audit["shared_variant_keys"]
        .gt(0)
        .sum()
    )
)

print("\nCases with gene-annotation conflicts:")
print(
    int(
        multi_maf_audit["gene_annotation_conflicts"]
        .gt(0)
        .sum()
    )
)

print("\nCases with classification-annotation conflicts:")
print(
    int(
        multi_maf_audit["classification_annotation_conflicts"]
        .gt(0)
        .sum()
    )
)

print("\nCross-file overlap summary:")
print(
    multi_maf_audit[
        [
            "total_variant_rows",
            "unique_variant_keys",
            "shared_variant_keys",
            "keys_present_in_all_files",
        ]
    ].describe()
)

Multi-MAF cases audited: 215

Number of retained MAFs per case:
n_files
2    204
3      5
4      4
5      2
Name: count, dtype: int64

Cases with at least one shared genomic variant:
206

Cases with gene-annotation conflicts:
0

Cases with classification-annotation conflicts:
0

Cross-file overlap summary:
       total_variant_rows  unique_variant_keys  shared_variant_keys  \
count          215.000000           215.000000           215.000000   
mean           706.353488           396.051163           287.920930   
std           1799.112973          1006.583238           779.563164   
min              0.000000             0.000000             0.000000   
25%            116.500000            70.000000            41.000000   
50%            170.000000           107.000000            65.000000   
75%            476.500000           267.000000           193.000000   
max          16793.000000          8970.000000          7823.000000   

       keys_present_in_all_files  
count            

In [11]:
# =============================================================================
# Audit structural payload states by MAF multiplicity
# =============================================================================

payload_multiplicity_audit = case_payload_status.copy()

payload_multiplicity_audit["maf_multiplicity"] = (
    payload_multiplicity_audit["n_mafs"]
    .astype("int64")
    .gt(1)
    .map(
        {
            False: "single_maf",
            True: "multi_maf",
        }
    )
)

structural_status_summary = (
    payload_multiplicity_audit.groupby(
        "maf_multiplicity",
        observed=True,
    )
    .agg(
        n_cases=("tcga_case_barcode", "size"),
        all_empty_cases=("all_mafs_empty", "sum"),
        mixed_empty_nonempty_cases=("mixed_empty_nonempty", "sum"),
        cases_with_variant_rows=(
            "n_mafs_with_variant_rows",
            lambda values: int((values.astype("int64") > 0).sum()),
        ),
    )
)

print(structural_status_summary)

print("\nMulti-MAF structural-status combinations:")
print(
    payload_multiplicity_audit.loc[
        payload_multiplicity_audit["maf_multiplicity"].eq("multi_maf"),
        [
            "n_mafs",
            "n_mafs_with_variant_rows",
            "n_empty_mafs",
            "all_mafs_empty",
            "mixed_empty_nonempty",
        ],
    ]
    .value_counts()
    .sort_index()
)

                  n_cases  all_empty_cases  mixed_empty_nonempty_cases  \
maf_multiplicity                                                         
multi_maf             215                1                           5   
single_maf           8977               57                           0   

                  cases_with_variant_rows  
maf_multiplicity                           
multi_maf                             214  
single_maf                           8920  

Multi-MAF structural-status combinations:
n_mafs  n_mafs_with_variant_rows  n_empty_mafs  all_mafs_empty  mixed_empty_nonempty
2       0                         2             True            False                     1
        1                         1             False           True                      4
        2                         0             False           False                   199
3       1                         2             False           True                      1
        3                      

## Frozen case-level mutation representation

Case-level mutation states follow the prespecified rules documented in
`docs/contracts/phase4b/PHASE4B_450_ANALYSIS_CONTRACT.md`.

For primary analysis:

- multiple exact-sample MAFs are combined by union followed by genomic-key
  deduplication;
- cases with all retained MAFs structurally empty remain indeterminate;
- mixed empty/non-empty cases remain informative through their non-empty MAFs;
- gene states are reported as
  `qualifying_somatic_variant_observed` or
  `no_qualifying_somatic_variant_observed`, not as absolute mutant/wild-type
  states.

A single-MAF restriction is retained as a prespecified sensitivity analysis.

## Frozen qualifying somatic-variant definition

Primary gene-level mutation status is based on the prespecified
`Variant_Classification` whitelist documented in
`docs/contracts/phase4b/PHASE4B_450_ANALYSIS_CONTRACT.md`.

The primary definition includes protein-altering variants and canonical
`Splice_Site` events. `Splice_Region` is excluded from the primary definition
and evaluated separately in a prespecified sensitivity analysis.

No post hoc VAF, depth, pathogenicity, driver, or external functional-score
filters are introduced.


In [12]:
# =============================================================================
# Construct primary qualifying case-level variant representation
# =============================================================================

PRIMARY_QUALIFYING_CLASSIFICATIONS = {
    "Missense_Mutation",
    "Nonsense_Mutation",
    "Frame_Shift_Del",
    "Frame_Shift_Ins",
    "Splice_Site",
    "In_Frame_Del",
    "In_Frame_Ins",
    "Translation_Start_Site",
    "Nonstop_Mutation",
}

QUALIFYING_MAF_FIELDS = VARIANT_KEY_FIELDS + [
    "Hugo_Symbol",
    "Variant_Classification",
]

qualifying_frames = []
raw_qualifying_rows = 0

for row in maf_file_table.itertuples(index=False):
    maf = pd.read_csv(
        row.maf_path,
        sep="\t",
        comment="#",
        usecols=QUALIFYING_MAF_FIELDS,
        dtype="string",
        low_memory=False,
    )

    if maf.empty:
        continue

    qualifying = maf.loc[
        maf["Variant_Classification"].isin(
            PRIMARY_QUALIFYING_CLASSIFICATIONS
        )
    ].copy()

    if qualifying.empty:
        continue

    raw_qualifying_rows += len(qualifying)

    qualifying["tcga_case_barcode"] = row.tcga_case_barcode
    qualifying["project_id"] = row.project_id
    qualifying["file_id"] = row.file_id

    qualifying_frames.append(qualifying)

qualifying_variants_raw = pd.concat(
    qualifying_frames,
    ignore_index=True,
)

CASE_VARIANT_KEY = [
    "tcga_case_barcode",
    *VARIANT_KEY_FIELDS,
]

qualifying_variants = (
    qualifying_variants_raw
    .drop_duplicates(
        subset=CASE_VARIANT_KEY,
        keep="first",
    )
    .reset_index(drop=True)
)

informative_cases = case_payload_status.loc[
    ~case_payload_status["all_mafs_empty"],
    "tcga_case_barcode",
]

missing_gene = (
    qualifying_variants["Hugo_Symbol"].isna()
    | qualifying_variants["Hugo_Symbol"].str.strip().eq("")
)

print(f"Raw qualifying variant rows: {raw_qualifying_rows:,}")
print(
    "Case-level unique qualifying variants:",
    f"{len(qualifying_variants):,}",
)
print(
    "Cross-file duplicate observations removed:",
    f"{raw_qualifying_rows - len(qualifying_variants):,}",
)
print(
    "Informative mutation-resource cases:",
    f"{informative_cases.nunique():,}",
)
print(
    "Cases with >=1 qualifying variant:",
    f"{qualifying_variants['tcga_case_barcode'].nunique():,}",
)
print(
    "Informative cases with no qualifying variant observed:",
    f"{informative_cases.nunique() - qualifying_variants['tcga_case_barcode'].nunique():,}",
)
print(
    "Qualifying variants with missing/blank Hugo_Symbol:",
    f"{int(missing_gene.sum()):,}",
)

Raw qualifying variant rows: 1,608,018
Case-level unique qualifying variants: 1,557,246
Cross-file duplicate observations removed: 50,772
Informative mutation-resource cases: 9,134
Cases with >=1 qualifying variant: 9,121
Informative cases with no qualifying variant observed: 13
Qualifying variants with missing/blank Hugo_Symbol: 0


In [13]:
# =============================================================================
# Construct case-gene observations and mutation-only prevalence summaries
# =============================================================================

informative_case_table = (
    case_payload_status.loc[
        ~case_payload_status["all_mafs_empty"],
        [
            "tcga_case_barcode",
            "project_id",
        ],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

case_gene_observed = (
    qualifying_variants[
        [
            "tcga_case_barcode",
            "project_id",
            "Hugo_Symbol",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

project_denominators = (
    informative_case_table
    .groupby("project_id", as_index=False)
    .agg(
        n_informative_cases=("tcga_case_barcode", "nunique"),
    )
)

gene_project_prevalence = (
    case_gene_observed
    .groupby(
        [
            "Hugo_Symbol",
            "project_id",
        ],
        as_index=False,
    )
    .agg(
        n_mutated_cases=("tcga_case_barcode", "nunique"),
    )
    .merge(
        project_denominators,
        on="project_id",
        how="left",
        validate="many_to_one",
    )
)

gene_project_prevalence["mutation_prevalence"] = (
    gene_project_prevalence["n_mutated_cases"]
    / gene_project_prevalence["n_informative_cases"]
)

gene_overall_summary = (
    case_gene_observed
    .groupby("Hugo_Symbol", as_index=False)
    .agg(
        n_mutated_cases=("tcga_case_barcode", "nunique"),
        n_projects_observed=("project_id", "nunique"),
    )
)

print(f"Informative cases: {len(informative_case_table):,}")
print(f"Unique case-gene observations: {len(case_gene_observed):,}")
print(f"Genes with >=1 qualifying observation: {case_gene_observed['Hugo_Symbol'].nunique():,}")
print(f"Projects represented: {informative_case_table['project_id'].nunique():,}")

print("\nOverall mutated-case count per gene:")
print(
    gene_overall_summary["n_mutated_cases"]
    .describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nProjects represented per gene:")
print(
    gene_overall_summary["n_projects_observed"]
    .describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

Informative cases: 9,134
Unique case-gene observations: 1,353,185
Genes with >=1 qualifying observation: 18,888
Projects represented: 33

Overall mutated-case count per gene:
count    18888.000000
mean        71.642577
std         76.055626
min          1.000000
50%         54.000000
75%         89.000000
90%        141.000000
95%        192.000000
99%        339.130000
max       3434.000000
Name: n_mutated_cases, dtype: float64

Projects represented per gene:
count    18888.000000
mean        15.469452
std          5.542160
min          1.000000
50%         16.000000
75%         19.000000
90%         22.000000
95%         24.000000
99%         27.000000
max         33.000000
Name: n_projects_observed, dtype: float64


In [14]:
# =============================================================================
# Calibrate mutation-only recurrent-gene eligibility thresholds
# =============================================================================

gene_project_testability = gene_project_prevalence.copy()

gene_project_testability["n_nonmutated_cases"] = (
    gene_project_testability["n_informative_cases"]
    - gene_project_testability["n_mutated_cases"]
)

threshold_grid = []

for min_mutated_per_project in [3, 5, 10]:
    for min_nonmutated_per_project in [10, 20]:
        project_support = (
            gene_project_testability.loc[
                (gene_project_testability["n_mutated_cases"] >= min_mutated_per_project)
                & (
                    gene_project_testability["n_nonmutated_cases"]
                    >= min_nonmutated_per_project
                )
            ]
            .groupby("Hugo_Symbol")
            .size()
        )

        eligible_genes = gene_overall_summary.loc[
            gene_overall_summary["n_mutated_cases"] >= 20,
            "Hugo_Symbol",
        ]

        n_eligible = int(
            eligible_genes.isin(
                project_support.loc[project_support >= 3].index
            ).sum()
        )

        threshold_grid.append(
            {
                "min_total_mutated_cases": 20,
                "min_mutated_per_project": min_mutated_per_project,
                "min_nonmutated_per_project": min_nonmutated_per_project,
                "min_supported_projects": 3,
                "n_eligible_genes": n_eligible,
            }
        )

threshold_grid = pd.DataFrame(threshold_grid)

print(threshold_grid.to_string(index=False))

 min_total_mutated_cases  min_mutated_per_project  min_nonmutated_per_project  min_supported_projects  n_eligible_genes
                      20                        3                          10                       3             15194
                      20                        3                          20                       3             15194
                      20                        5                          10                       3             11049
                      20                        5                          20                       3             11049
                      20                       10                          10                       3              4449
                      20                       10                          20                       3              4449


## Frozen primary recurrent-gene eligibility

The primary gene universe is defined from mutation prevalence and
project-level testability before program-score associations are inspected.

A gene is eligible when it has, in at least 3 TCGA projects:

- at least 10 informative mutated cases; and
- at least 20 informative non-mutated cases.

This outcome-blind rule retains 4,449 genes. Eligibility is independent of
association strength, biological annotation, pathway membership, or prior
expectation.

Full rationale and threshold provenance are documented in
`docs/contracts/phase4b/PHASE4B_450_ANALYSIS_CONTRACT.md`.

In [15]:
# =============================================================================
# Materialize frozen primary recurrent-gene universe
# =============================================================================

PRIMARY_MIN_MUTATED_PER_PROJECT = 10
PRIMARY_MIN_NONMUTATED_PER_PROJECT = 20
PRIMARY_MIN_SUPPORTED_PROJECTS = 3

primary_project_support = (
    gene_project_testability.loc[
        (
            gene_project_testability["n_mutated_cases"]
            >= PRIMARY_MIN_MUTATED_PER_PROJECT
        )
        & (
            gene_project_testability["n_nonmutated_cases"]
            >= PRIMARY_MIN_NONMUTATED_PER_PROJECT
        )
    ]
    .groupby("Hugo_Symbol", as_index=False)
    .agg(
        n_supported_projects=("project_id", "nunique"),
    )
)

primary_gene_universe = (
    gene_overall_summary
    .merge(
        primary_project_support,
        on="Hugo_Symbol",
        how="inner",
        validate="one_to_one",
    )
    .loc[
        lambda frame:
        frame["n_supported_projects"] >= PRIMARY_MIN_SUPPORTED_PROJECTS
    ]
    .sort_values(
        [
            "n_supported_projects",
            "n_mutated_cases",
            "Hugo_Symbol",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

print(f"Primary eligible genes: {len(primary_gene_universe):,}")
print(
    "Supported-project range:",
    f"{primary_gene_universe['n_supported_projects'].min()}–"
    f"{primary_gene_universe['n_supported_projects'].max()}",
)
print(
    "Mutated-case range:",
    f"{primary_gene_universe['n_mutated_cases'].min():,}–"
    f"{primary_gene_universe['n_mutated_cases'].max():,}",
)

assert len(primary_gene_universe) == 4449
assert (
    primary_gene_universe["n_supported_projects"]
    >= PRIMARY_MIN_SUPPORTED_PROJECTS
).all()

Primary eligible genes: 4,449
Supported-project range: 3–22
Mutated-case range: 49–3,434


## Frozen primary program universe and inferential family

Primary inference is restricted to the three frozen Phase 4 consensus tumor
programs:

- `CONSENSUS_TX_01`;
- `CONSENSUS_TX_02`;
- `CONSENSUS_TX_03`.

Every eligible gene is tested against every program, yielding one complete
primary family of 13,347 gene × program tests. Benjamini–Hochberg correction is
applied globally across this family.

Program identity, orientation, membership, and tumor scores remain exactly as
frozen upstream. Sensitivity and exploratory analyses remain outside the primary
FDR family.

Full inferential-family rules are documented in
`docs/contracts/phase4b/PHASE4B_450_ANALYSIS_CONTRACT.md`.

In [16]:
# =============================================================================
# Load frozen consensus tumor-score artifact
# =============================================================================

CONSENSUS_TUMOR_SCORE_ARTIFACT_ID = (
    "phase4.401.consensus_tumor_scores"
)

consensus_score_entry = artifact_registry["artifacts"][
    CONSENSUS_TUMOR_SCORE_ARTIFACT_ID
]

CONSENSUS_TUMOR_SCORE_PATH = (
    Paths.root / consensus_score_entry["path"]
)

if consensus_score_entry["status"] != "frozen":
    raise ValueError(
        "Consensus tumor-score artifact is not frozen."
    )

if not CONSENSUS_TUMOR_SCORE_PATH.is_file():
    raise FileNotFoundError(
        "Frozen consensus tumor-score artifact is unavailable locally."
    )

consensus_tumor_scores = pd.read_parquet(
    CONSENSUS_TUMOR_SCORE_PATH
)

print(
    "Artifact:",
    CONSENSUS_TUMOR_SCORE_ARTIFACT_ID,
)
print(
    "Path:",
    project_relative_path(CONSENSUS_TUMOR_SCORE_PATH),
)
print(
    "Status:",
    consensus_score_entry["status"],
)
print(
    "Shape:",
    consensus_tumor_scores.shape,
)
print(
    "\nColumns:"
)
print(consensus_tumor_scores.columns.tolist())

print("\nDtypes:")
print(consensus_tumor_scores.dtypes)

Artifact: phase4.401.consensus_tumor_scores
Path: data/processed/consensus_programs/401_consensus_tumor_scores.parquet
Status: frozen
Shape: (9965, 8)

Columns:
['final_sample_column_index', 'case_submitter_id', 'sample_submitter_id', 'project_id', 'methylation_platform', 'CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03']

Dtypes:
final_sample_column_index      int64
case_submitter_id                str
sample_submitter_id              str
project_id                       str
methylation_platform             str
CONSENSUS_TX_01              float64
CONSENSUS_TX_02              float64
CONSENSUS_TX_03              float64
dtype: object


In [17]:
# =============================================================================
# Validate frozen consensus-score cohort integrity
# =============================================================================

PROGRAM_COLUMNS = [
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
]

score_case_duplicates = (
    consensus_tumor_scores["case_submitter_id"]
    .duplicated()
    .sum()
)

score_missingness = (
    consensus_tumor_scores[PROGRAM_COLUMNS]
    .isna()
    .sum()
)

mutation_score_alignment = (
    informative_case_table
    .merge(
        consensus_tumor_scores[
            [
                "case_submitter_id",
                "project_id",
                *PROGRAM_COLUMNS,
            ]
        ],
        left_on="tcga_case_barcode",
        right_on="case_submitter_id",
        how="left",
        validate="one_to_one",
        suffixes=("_mutation", "_score"),
    )
)

missing_score_cases = (
    mutation_score_alignment["case_submitter_id"]
    .isna()
    .sum()
)

project_mismatches = (
    mutation_score_alignment["case_submitter_id"].notna()
    & (
        mutation_score_alignment["project_id_mutation"]
        != mutation_score_alignment["project_id_score"]
    )
).sum()

print(
    "Duplicate consensus-score cases:",
    f"{int(score_case_duplicates):,}",
)

print("\nMissing program scores:")
print(score_missingness)

print(
    "\nInformative mutation cases:",
    f"{len(informative_case_table):,}",
)
print(
    "Informative cases without consensus scores:",
    f"{int(missing_score_cases):,}",
)
print(
    "Project-ID mismatches after case join:",
    f"{int(project_mismatches):,}",
)

Duplicate consensus-score cases: 0

Missing program scores:
CONSENSUS_TX_01    0
CONSENSUS_TX_02    0
CONSENSUS_TX_03    0
dtype: int64

Informative mutation cases: 9,134
Informative cases without consensus scores: 0
Project-ID mismatches after case join: 0


In [18]:
# =============================================================================
# Inspect frozen consensus-score scale and within-project variability
# =============================================================================

score_summary = (
    consensus_tumor_scores[PROGRAM_COLUMNS]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
    .T
)

within_project_sd = (
    consensus_tumor_scores
    .groupby("project_id")[PROGRAM_COLUMNS]
    .std()
)

within_project_sd_summary = (
    within_project_sd
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .T
)

zero_variance_projects = (
    within_project_sd
    .eq(0)
    .sum()
)

print("Global frozen-score distributions:")
print(score_summary)

print("\nWithin-project SD distributions:")
print(within_project_sd_summary)

print("\nProjects with zero within-project variance:")
print(zero_variance_projects)

Global frozen-score distributions:
                  count          mean  std       min        1%        5%  \
CONSENSUS_TX_01  9965.0 -7.986030e-17  1.0 -2.587496 -1.703455 -1.249784   
CONSENSUS_TX_02  9965.0  1.483120e-16  1.0 -3.491363 -2.533586 -1.837564   
CONSENSUS_TX_03  9965.0  1.825378e-16  1.0 -3.161696 -2.311652 -1.671706   

                      25%       50%       75%       95%       99%       max  
CONSENSUS_TX_01 -0.650590 -0.153567  0.454833  1.746243  3.935073  5.838075  
CONSENSUS_TX_02 -0.545140  0.002015  0.596561  1.777248  1.998795  2.315918  
CONSENSUS_TX_03 -0.684095  0.017851  0.684514  1.537462  2.546836  4.659136  

Within-project SD distributions:
                 count      mean       std       min       10%       25%  \
CONSENSUS_TX_01   33.0  0.717713  0.192323  0.395350  0.458466  0.618741   
CONSENSUS_TX_02   33.0  0.393857  0.185647  0.167238  0.197662  0.275668   
CONSENSUS_TX_03   33.0  0.648190  0.178216  0.269842  0.419849  0.512570   

         

## Frozen program-score scale

Consensus tumor scores are used exactly as frozen in Phase 4, without additional
global or within-project restandardization.

The primary mutation-status coefficient is therefore interpreted in frozen
consensus-program score units.

Detailed scale and interpretation rules are documented in
`docs/contracts/phase4b/PHASE4B_450_ANALYSIS_CONTRACT.md`.

In [19]:
# =============================================================================
# Inspect frozen tumor confounder-covariate artifact
# =============================================================================

CONFOUNDER_ARTIFACT_ID = (
    "phase2.204.confounder_covariates"
)

confounder_entry = artifact_registry["artifacts"][
    CONFOUNDER_ARTIFACT_ID
]

CONFOUNDER_PATH = (
    Paths.root / confounder_entry["path"]
)

if confounder_entry["status"] != "frozen":
    raise ValueError(
        "Tumor confounder-covariate artifact is not frozen."
    )

if not CONFOUNDER_PATH.is_file():
    raise FileNotFoundError(
        "Frozen tumor confounder-covariate artifact is unavailable locally."
    )

if CONFOUNDER_PATH.suffix == ".parquet":
    confounder_covariates = pd.read_parquet(
        CONFOUNDER_PATH
    )
elif CONFOUNDER_PATH.suffix == ".csv":
    confounder_covariates = pd.read_csv(
        CONFOUNDER_PATH
    )
else:
    raise ValueError(
        f"Unsupported confounder artifact format: {CONFOUNDER_PATH.suffix}"
    )

print("Artifact:", CONFOUNDER_ARTIFACT_ID)
print(
    "Path:",
    project_relative_path(CONFOUNDER_PATH),
)
print("Status:", confounder_entry["status"])
print("Shape:", confounder_covariates.shape)

print("\nColumns:")
print(confounder_covariates.columns.tolist())

print("\nDtypes:")
print(confounder_covariates.dtypes)

Artifact: phase2.204.confounder_covariates
Path: data/interim/metadata/tcga_primary_tumor_multiomic_confounder_covariates.csv
Status: frozen
Shape: (9965, 18)

Columns:
['final_sample_column_index', 'case_submitter_id', 'sample_submitter_id', 'project_id', 'methylation_platform', 'absolute_call_status', 'absolute_purity', 'consensus_purity_estimate', 'purity_availability', 'leukocyte_fraction', 'external_panimmune_proliferation_score', 'gene_assigned_fraction_of_accounted', 'missing_beta_fraction', 'tissue_source_site', 'rna_plate', 'rna_center', 'methylation_plate', 'sex_at_birth']

Dtypes:
final_sample_column_index                   int64
case_submitter_id                             str
sample_submitter_id                           str
project_id                                    str
methylation_platform                          str
absolute_call_status                          str
absolute_purity                           float64
consensus_purity_estimate                 float64
p

In [20]:
# =============================================================================
# Audit candidate biological-covariate availability
# =============================================================================

CANDIDATE_BIOLOGICAL_COVARIATES = [
    "absolute_purity",
    "consensus_purity_estimate",
    "leukocyte_fraction",
    "external_panimmune_proliferation_score",
]

mutation_confounder_alignment = (
    informative_case_table
    .merge(
        confounder_covariates[
            [
                "case_submitter_id",
                "project_id",
                "absolute_call_status",
                "purity_availability",
                *CANDIDATE_BIOLOGICAL_COVARIATES,
            ]
        ],
        left_on="tcga_case_barcode",
        right_on="case_submitter_id",
        how="left",
        validate="one_to_one",
        suffixes=("_mutation", "_covariate"),
    )
)

missing_confounder_rows = (
    mutation_confounder_alignment["case_submitter_id"]
    .isna()
    .sum()
)

project_mismatches = (
    mutation_confounder_alignment["case_submitter_id"].notna()
    & (
        mutation_confounder_alignment["project_id_mutation"]
        != mutation_confounder_alignment["project_id_covariate"]
    )
).sum()

covariate_availability = pd.DataFrame(
    {
        "nonmissing_cases": mutation_confounder_alignment[
            CANDIDATE_BIOLOGICAL_COVARIATES
        ].notna().sum(),
        "missing_cases": mutation_confounder_alignment[
            CANDIDATE_BIOLOGICAL_COVARIATES
        ].isna().sum(),
    }
)

covariate_availability["coverage_fraction"] = (
    covariate_availability["nonmissing_cases"]
    / len(mutation_confounder_alignment)
)

print(
    "Informative mutation cases without confounder record:",
    f"{int(missing_confounder_rows):,}",
)
print(
    "Project-ID mismatches:",
    f"{int(project_mismatches):,}",
)

print("\nBiological-covariate availability:")
print(covariate_availability)

print("\nPurity availability categories:")
print(
    mutation_confounder_alignment["purity_availability"]
    .value_counts(dropna=False)
)

print("\nABSOLUTE call-status categories:")
print(
    mutation_confounder_alignment["absolute_call_status"]
    .value_counts(dropna=False)
)

Informative mutation cases without confounder record: 0
Project-ID mismatches: 0

Biological-covariate availability:
                                        nonmissing_cases  missing_cases  \
absolute_purity                                     8806            328   
consensus_purity_estimate                           7395           1739   
leukocyte_fraction                                  8891            243   
external_panimmune_proliferation_score              8369            765   

                                        coverage_fraction  
absolute_purity                                  0.964090  
consensus_purity_estimate                        0.809612  
leukocyte_fraction                               0.973396  
external_panimmune_proliferation_score           0.916247  

Purity availability categories:
purity_availability
CPE and ABSOLUTE    7176
ABSOLUTE only       1630
CPE only             219
Neither              109
Name: count, dtype: int64

ABSOLUTE call-status catego

In [21]:
# =============================================================================
# Audit biological-covariate coverage by TCGA project
# =============================================================================

project_covariate_coverage = (
    mutation_confounder_alignment
    .groupby("project_id_mutation", as_index=False)
    .agg(
        n_cases=("tcga_case_barcode", "size"),
        absolute_purity_available=(
            "absolute_purity",
            lambda values: int(values.notna().sum()),
        ),
        consensus_purity_available=(
            "consensus_purity_estimate",
            lambda values: int(values.notna().sum()),
        ),
        leukocyte_fraction_available=(
            "leukocyte_fraction",
            lambda values: int(values.notna().sum()),
        ),
        proliferation_available=(
            "external_panimmune_proliferation_score",
            lambda values: int(values.notna().sum()),
        ),
    )
)

for column in [
    "absolute_purity_available",
    "consensus_purity_available",
    "leukocyte_fraction_available",
    "proliferation_available",
]:
    coverage_column = column.replace("_available", "_coverage")
    project_covariate_coverage[coverage_column] = (
        project_covariate_coverage[column]
        / project_covariate_coverage["n_cases"]
    )

coverage_columns = [
    "absolute_purity_coverage",
    "consensus_purity_coverage",
    "leukocyte_fraction_coverage",
    "proliferation_coverage",
]

print("Per-project coverage summary:")
print(
    project_covariate_coverage[
        coverage_columns
    ].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nLowest absolute-purity coverage projects:")
print(
    project_covariate_coverage[
        [
            "project_id_mutation",
            "n_cases",
            "absolute_purity_coverage",
        ]
    ]
    .sort_values("absolute_purity_coverage")
    .head(10)
    .to_string(index=False)
)

print("\nLowest proliferation-score coverage projects:")
print(
    project_covariate_coverage[
        [
            "project_id_mutation",
            "n_cases",
            "proliferation_coverage",
        ]
    ]
    .sort_values("proliferation_coverage")
    .head(10)
    .to_string(index=False)
)

Per-project coverage summary:
       absolute_purity_coverage  consensus_purity_coverage  \
count                 33.000000                  33.000000   
mean                   0.960945                   0.633412   
std                    0.043193                   0.486268   
min                    0.810345                   0.000000   
10%                    0.900919                   0.000000   
25%                    0.958290                   0.000000   
50%                    0.974359                   0.992958   
75%                    0.985447                   0.998028   
90%                    1.000000                   1.000000   
max                    1.000000                   1.000000   

       leukocyte_fraction_coverage  proliferation_coverage  
count                    33.000000               33.000000  
mean                      0.904937                0.850910  
std                       0.291374                0.290164  
min                       0.000000         

## Frozen covariate-adjustment policy

The primary model adjusts for TCGA project/lineage only.

Two prespecified complete-case sensitivity analyses additionally evaluate:

- `absolute_purity`; and
- `external_panimmune_proliferation_score`.

Purity and proliferation are not added to the primary model because their
coverage is incomplete and lineage dependent, and proliferation adjustment may
also change the biological estimand.

`leukocyte_fraction`, technical covariates, and `sex_at_birth` are not included
by default.

The complete covariate rationale and sensitivity boundaries are documented in
`docs/contracts/phase4b/PHASE4B_450_ANALYSIS_CONTRACT.md`.

## Prespecified statistical analysis plan

For each eligible gene × program pair, the primary model is:

`program_score ~ mutation_status + C(project_id)`

using HC3 standard errors and only the gene's supported TCGA projects.

Primary p-values are adjusted jointly across all 13,347 tests using
Benjamini–Hochberg FDR (`q < 0.05`).

Primary FDR-supported associations are then characterized by within-project
effect direction and leave-one-project-out stability. The
`cross_cancer_recurrent` designation additionally requires:

- at least 3 supported projects;
- at least 3 projects with the pooled effect direction;
- at least 70% directional concordance; and
- no leave-one-project-out sign reversal.

Sensitivity analyses characterize stability but do not redefine primary
significance or recurrence. Full statistical and interpretation rules are
documented in `docs/contracts/phase4b/PHASE4B_450_ANALYSIS_CONTRACT.md`.

In [22]:
# =============================================================================
# Construct mutation-resource availability audit cohort
# =============================================================================

mutation_availability = (
    case_eligibility[
        [
            "tcga_case_barcode",
            "project_id",
            "mutation_resource_status",
            "mutation_resource_eligible",
        ]
    ]
    .merge(
        case_payload_status[
            [
                "tcga_case_barcode",
                "all_mafs_empty",
                "mixed_empty_nonempty",
            ]
        ],
        on="tcga_case_barcode",
        how="left",
        validate="one_to_one",
    )
    .merge(
        consensus_tumor_scores[
            [
                "case_submitter_id",
                *PROGRAM_COLUMNS,
            ]
        ],
        left_on="tcga_case_barcode",
        right_on="case_submitter_id",
        how="left",
        validate="one_to_one",
    )
)

mutation_availability["analysis_availability"] = "unavailable"

eligible = mutation_availability[
    "mutation_resource_eligible"
].eq("True")

all_empty = (
    eligible
    & mutation_availability["all_mafs_empty"].fillna(False)
)

informative = eligible & ~all_empty

mutation_availability.loc[
    all_empty,
    "analysis_availability",
] = "indeterminate_all_empty"

mutation_availability.loc[
    informative,
    "analysis_availability",
] = "informative"

print("Cohort cases:", f"{len(mutation_availability):,}")

print("\nAnalysis-availability status:")
print(
    mutation_availability["analysis_availability"]
    .value_counts()
)

print("\nFrozen mutation-resource status:")
print(
    mutation_availability["mutation_resource_status"]
    .value_counts(dropna=False)
)

print("\nCases missing consensus scores:")
print(
    mutation_availability[PROGRAM_COLUMNS]
    .isna()
    .any(axis=1)
    .sum()
)

Cohort cases: 9,965

Analysis-availability status:
analysis_availability
informative                9134
unavailable                 773
indeterminate_all_empty      58
Name: count, dtype: int64

Frozen mutation-resource status:
mutation_resource_status
eligible_single_exact_sample_maf       8977
no_mutation_resource                    735
eligible_multiple_exact_sample_mafs     215
no_exact_frozen_sample_match             38
Name: count, dtype: int64[pyarrow]

Cases missing consensus scores:
0


In [23]:
# =============================================================================
# Audit within-project selection by mutation-resource availability
# =============================================================================

mutation_availability["primary_analysis_included"] = (
    mutation_availability["analysis_availability"].eq("informative")
)

selection_audit_rows = []

for project_id, project_frame in mutation_availability.groupby(
    "project_id",
    sort=True,
):
    included = project_frame.loc[
        project_frame["primary_analysis_included"]
    ]
    excluded = project_frame.loc[
        ~project_frame["primary_analysis_included"]
    ]

    for program in PROGRAM_COLUMNS:
        included_values = included[program].dropna()
        excluded_values = excluded[program].dropna()

        row = {
            "project_id": project_id,
            "program": program,
            "n_included": len(included_values),
            "n_excluded": len(excluded_values),
            "mean_included": included_values.mean(),
            "mean_excluded": excluded_values.mean(),
            "mean_difference": (
                included_values.mean()
                - excluded_values.mean()
                if len(excluded_values) > 0
                else float("nan")
            ),
            "standardized_mean_difference": float("nan"),
        }

        if len(included_values) >= 2 and len(excluded_values) >= 2:
            pooled_sd = (
                (
                    included_values.var(ddof=1)
                    + excluded_values.var(ddof=1)
                )
                / 2
            ) ** 0.5

            if pooled_sd > 0:
                row["standardized_mean_difference"] = (
                    row["mean_difference"] / pooled_sd
                )

        selection_audit_rows.append(row)

selection_audit = pd.DataFrame(selection_audit_rows)

stable_selection_audit = selection_audit.loc[
    (selection_audit["n_included"] >= 5)
    & (selection_audit["n_excluded"] >= 5)
].copy()

print(
    "Projects with >=5 included and >=5 excluded cases:",
    stable_selection_audit["project_id"].nunique(),
)

print("\nWithin-project standardized mean-difference summary:")
print(
    stable_selection_audit.groupby("program")[
        "standardized_mean_difference"
    ].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nLargest absolute standardized differences:")
print(
    stable_selection_audit.assign(
        abs_smd=lambda frame:
        frame["standardized_mean_difference"].abs()
    )
    .sort_values("abs_smd", ascending=False)
    [
        [
            "project_id",
            "program",
            "n_included",
            "n_excluded",
            "standardized_mean_difference",
        ]
    ]
    .head(15)
    .to_string(index=False)
)

Projects with >=5 included and >=5 excluded cases: 24

Within-project standardized mean-difference summary:
                 count      mean       std       min       10%       25%  \
program                                                                    
CONSENSUS_TX_01   24.0 -0.042153  0.343346 -0.670542 -0.427145 -0.394062   
CONSENSUS_TX_02   24.0 -0.051319  0.453518 -0.837378 -0.601343 -0.318935   
CONSENSUS_TX_03   24.0 -0.083340  0.353758 -0.942954 -0.529516 -0.229198   

                      50%       75%       90%       max  
program                                                  
CONSENSUS_TX_01  0.013198  0.128224  0.358376  0.684483  
CONSENSUS_TX_02 -0.048858  0.182766  0.559518  0.899287  
CONSENSUS_TX_03 -0.028140  0.159283  0.250722  0.550927  

Largest absolute standardized differences:
project_id         program  n_included  n_excluded  standardized_mean_difference
 TCGA-LIHC CONSENSUS_TX_03         362           9                     -0.942954
 TCGA-PRAD CONS

In [24]:
# =============================================================================
# Materialize primary gene-project support and analysis base cohort
# =============================================================================

primary_supported_gene_projects = (
    gene_project_testability.loc[
        (
            gene_project_testability["n_mutated_cases"]
            >= PRIMARY_MIN_MUTATED_PER_PROJECT
        )
        & (
            gene_project_testability["n_nonmutated_cases"]
            >= PRIMARY_MIN_NONMUTATED_PER_PROJECT
        )
        & (
            gene_project_testability["Hugo_Symbol"]
            .isin(primary_gene_universe["Hugo_Symbol"])
        )
    ]
    [
        [
            "Hugo_Symbol",
            "project_id",
            "n_mutated_cases",
            "n_nonmutated_cases",
            "n_informative_cases",
            "mutation_prevalence",
        ]
    ]
    .sort_values(
        [
            "Hugo_Symbol",
            "project_id",
        ]
    )
    .reset_index(drop=True)
)

primary_analysis_base = (
    informative_case_table
    .merge(
        consensus_tumor_scores[
            [
                "case_submitter_id",
                "sample_submitter_id",
                "project_id",
                *PROGRAM_COLUMNS,
            ]
        ],
        left_on="tcga_case_barcode",
        right_on="case_submitter_id",
        how="inner",
        validate="one_to_one",
        suffixes=("_mutation", "_score"),
    )
)

assert (
    primary_analysis_base["project_id_mutation"]
    == primary_analysis_base["project_id_score"]
).all()

primary_analysis_base = (
    primary_analysis_base
    .drop(columns=["project_id_score"])
    .rename(columns={"project_id_mutation": "project_id"})
)

support_counts = (
    primary_supported_gene_projects
    .groupby("Hugo_Symbol")["project_id"]
    .nunique()
)

print(
    "Primary analysis base cases:",
    f"{len(primary_analysis_base):,}",
)
print(
    "Primary genes:",
    f"{primary_supported_gene_projects['Hugo_Symbol'].nunique():,}",
)
print(
    "Eligible gene-project combinations:",
    f"{len(primary_supported_gene_projects):,}",
)
print(
    "Supported projects per gene:",
    f"{support_counts.min()}–{support_counts.max()}",
)

print("\nSupported-project count distribution:")
print(
    support_counts.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

Primary analysis base cases: 9,134
Primary genes: 4,449
Eligible gene-project combinations: 22,356
Supported projects per gene: 3–22

Supported-project count distribution:
count    4449.000000
mean        5.024949
std         2.372004
min         3.000000
25%         3.000000
50%         4.000000
75%         6.000000
90%         8.000000
95%        10.000000
99%        13.520000
max        22.000000
Name: project_id, dtype: float64


In [25]:
# =============================================================================
# Prepare sparse primary gene-level analysis indices
# =============================================================================

mutated_cases_by_gene = (
    case_gene_observed.loc[
        case_gene_observed["Hugo_Symbol"].isin(
            primary_gene_universe["Hugo_Symbol"]
        )
    ]
    .groupby("Hugo_Symbol")["tcga_case_barcode"]
    .apply(set)
    .to_dict()
)

supported_projects_by_gene = (
    primary_supported_gene_projects
    .groupby("Hugo_Symbol")["project_id"]
    .apply(tuple)
    .to_dict()
)

gene_analysis_sizes = []

for gene in primary_gene_universe["Hugo_Symbol"]:
    supported_projects = supported_projects_by_gene[gene]

    gene_cases = primary_analysis_base.loc[
        primary_analysis_base["project_id"].isin(
            supported_projects
        ),
        [
            "tcga_case_barcode",
            "project_id",
        ],
    ].copy()

    mutated_cases = mutated_cases_by_gene[gene]

    gene_cases["mutation_status"] = (
        gene_cases["tcga_case_barcode"]
        .isin(mutated_cases)
        .astype("int8")
    )

    gene_analysis_sizes.append(
        {
            "Hugo_Symbol": gene,
            "n_projects": gene_cases["project_id"].nunique(),
            "n_cases": len(gene_cases),
            "n_mutated": int(gene_cases["mutation_status"].sum()),
            "n_nonmutated": int(
                (gene_cases["mutation_status"] == 0).sum()
            ),
        }
    )

gene_analysis_sizes = pd.DataFrame(gene_analysis_sizes)

print(
    "Genes prepared:",
    f"{len(gene_analysis_sizes):,}",
)

print(
    "Genes violating minimum supported-project count:",
    int(
        (
            gene_analysis_sizes["n_projects"]
            < PRIMARY_MIN_SUPPORTED_PROJECTS
        ).sum()
    ),
)

print(
    "Genes with fewer than 30 total mutated cases:",
    int(
        (
            gene_analysis_sizes["n_mutated"]
            < (
                PRIMARY_MIN_MUTATED_PER_PROJECT
                * PRIMARY_MIN_SUPPORTED_PROJECTS
            )
        ).sum()
    ),
)

print("\nPrimary effective analysis-size summary:")
print(
    gene_analysis_sizes[
        [
            "n_projects",
            "n_cases",
            "n_mutated",
            "n_nonmutated",
        ]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

Genes prepared: 4,449
Genes violating minimum supported-project count: 0
Genes with fewer than 30 total mutated cases: 0

Primary effective analysis-size summary:
        n_projects      n_cases    n_mutated  n_nonmutated
count  4449.000000  4449.000000  4449.000000   4449.000000
mean      5.024949  2355.890987   108.936840   2246.954147
std       2.372004  1077.837137   106.592376    997.826089
min       3.000000  1007.000000    30.000000    792.000000
25%       3.000000  1431.000000    57.000000   1387.000000
50%       4.000000  1912.000000    79.000000   1846.000000
75%       6.000000  2813.000000   125.000000   2706.000000
90%       8.000000  4172.000000   197.200000   3935.600000
95%      10.000000  4534.000000   265.000000   4276.200000
99%      13.520000  5533.000000   489.040000   5061.120000
max      22.000000  8212.000000  3088.000000   6384.000000


In [27]:
# =============================================================================
# Define deterministic primary association model
# =============================================================================

import statsmodels.api as sm


def fit_primary_gene_program_model(
    gene: str,
    program: str,
) -> dict:
    """Fit the prespecified project-adjusted OLS model with HC3 uncertainty."""

    supported_projects = supported_projects_by_gene[gene]

    model_data = primary_analysis_base.loc[
        primary_analysis_base["project_id"].isin(supported_projects),
        [
            "tcga_case_barcode",
            "project_id",
            program,
        ],
    ].copy()

    model_data["mutation_status"] = (
        model_data["tcga_case_barcode"]
        .isin(mutated_cases_by_gene[gene])
        .astype(float)
    )

    project_dummies = pd.get_dummies(
        model_data["project_id"],
        prefix="project",
        drop_first=True,
        dtype=float,
    )

    design_matrix = pd.concat(
        [
            model_data[["mutation_status"]],
            project_dummies,
        ],
        axis=1,
    )

    design_matrix = sm.add_constant(
        design_matrix,
        has_constant="add",
    )

    model = sm.OLS(
        model_data[program].astype(float),
        design_matrix,
    ).fit(
        cov_type="HC3"
    )

    confidence_interval = model.conf_int().loc[
        "mutation_status"
    ]

    return {
        "Hugo_Symbol": gene,
        "program": program,
        "n_cases": len(model_data),
        "n_projects": model_data["project_id"].nunique(),
        "n_mutated": int(model_data["mutation_status"].sum()),
        "n_nonmutated": int(
            (model_data["mutation_status"] == 0).sum()
        ),
        "beta_mutation": float(
            model.params["mutation_status"]
        ),
        "se_hc3": float(
            model.bse["mutation_status"]
        ),
        "ci95_lower": float(
            confidence_interval.iloc[0]
        ),
        "ci95_upper": float(
            confidence_interval.iloc[1]
        ),
        "t_hc3": float(
            model.tvalues["mutation_status"]
        ),
        "p_value": float(
            model.pvalues["mutation_status"]
        ),
    }

In [29]:
# =============================================================================
# Smoke-test primary model implementation without inspecting association results
# =============================================================================

SMOKE_TEST_GENE = primary_gene_universe.loc[0, "Hugo_Symbol"]
SMOKE_TEST_PROGRAM = PROGRAM_COLUMNS[0]

smoke_test_result = fit_primary_gene_program_model(
    gene=SMOKE_TEST_GENE,
    program=SMOKE_TEST_PROGRAM,
)

expected_sizes = (
    gene_analysis_sizes
    .set_index("Hugo_Symbol")
    .loc[SMOKE_TEST_GENE]
)

assert smoke_test_result["n_cases"] == int(expected_sizes["n_cases"])
assert smoke_test_result["n_projects"] == int(expected_sizes["n_projects"])
assert smoke_test_result["n_mutated"] == int(expected_sizes["n_mutated"])
assert smoke_test_result["n_nonmutated"] == int(expected_sizes["n_nonmutated"])

for field in [
    "beta_mutation",
    "se_hc3",
    "ci95_lower",
    "ci95_upper",
    "t_hc3",
    "p_value",
]:
    assert pd.notna(smoke_test_result[field])

assert 0.0 <= smoke_test_result["p_value"] <= 1.0
assert smoke_test_result["se_hc3"] > 0

print("Primary-model smoke test passed.")
print("Gene:", SMOKE_TEST_GENE)
print("Program:", SMOKE_TEST_PROGRAM)
print("Cases:", smoke_test_result["n_cases"])
print("Projects:", smoke_test_result["n_projects"])
print("Mutated:", smoke_test_result["n_mutated"])
print("Non-mutated:", smoke_test_result["n_nonmutated"])

Primary-model smoke test passed.
Gene: TTN
Program: CONSENSUS_TX_01
Cases: 8212
Projects: 22
Mutated: 2423
Non-mutated: 5789


In [30]:
# =============================================================================
# Fit complete primary gene-program association family
# =============================================================================

primary_association_results = []

n_genes = len(primary_gene_universe)
n_expected_tests = n_genes * len(PROGRAM_COLUMNS)

for gene_index, gene in enumerate(
    primary_gene_universe["Hugo_Symbol"],
    start=1,
):
    for program in PROGRAM_COLUMNS:
        primary_association_results.append(
            fit_primary_gene_program_model(
                gene=gene,
                program=program,
            )
        )

    if gene_index % 250 == 0 or gene_index == n_genes:
        print(
            f"Processed genes: {gene_index:,}/{n_genes:,}"
        )

primary_associations = pd.DataFrame(
    primary_association_results
)

assert len(primary_associations) == n_expected_tests
assert primary_associations["p_value"].notna().all()
assert primary_associations["beta_mutation"].notna().all()
assert primary_associations["se_hc3"].gt(0).all()

print(
    "\nPrimary association tests:",
    f"{len(primary_associations):,}",
)
print(
    "Unique genes:",
    f"{primary_associations['Hugo_Symbol'].nunique():,}",
)
print(
    "Programs:",
    f"{primary_associations['program'].nunique():,}",
)
print(
    "Missing p-values:",
    f"{primary_associations['p_value'].isna().sum():,}",
)

Processed genes: 250/4,449
Processed genes: 500/4,449
Processed genes: 750/4,449
Processed genes: 1,000/4,449
Processed genes: 1,250/4,449
Processed genes: 1,500/4,449
Processed genes: 1,750/4,449
Processed genes: 2,000/4,449
Processed genes: 2,250/4,449
Processed genes: 2,500/4,449
Processed genes: 2,750/4,449
Processed genes: 3,000/4,449
Processed genes: 3,250/4,449
Processed genes: 3,500/4,449
Processed genes: 3,750/4,449
Processed genes: 4,000/4,449
Processed genes: 4,250/4,449
Processed genes: 4,449/4,449

Primary association tests: 13,347
Unique genes: 4,449
Programs: 3
Missing p-values: 0


In [31]:
# =============================================================================
# Apply BH-FDR correction to complete primary inferential family
# =============================================================================

from statsmodels.stats.multitest import multipletests


reject_fdr, q_values, _, _ = multipletests(
    primary_associations["p_value"].to_numpy(),
    alpha=0.05,
    method="fdr_bh",
)

primary_associations["q_value"] = q_values
primary_associations["primary_fdr_supported"] = reject_fdr

assert primary_associations["q_value"].between(0, 1).all()
assert (
    primary_associations["primary_fdr_supported"]
    == primary_associations["q_value"].lt(0.05)
).all()

print(
    "Primary tests:",
    f"{len(primary_associations):,}",
)
print(
    "BH-FDR supported tests (q < 0.05):",
    f"{int(primary_associations['primary_fdr_supported'].sum()):,}",
)

print("\nBH-FDR supported tests by program:")
print(
    primary_associations
    .groupby("program")["primary_fdr_supported"]
    .agg(
        n_tests="size",
        n_fdr_supported="sum",
    )
)

print("\nQ-value distribution:")
print(
    primary_associations["q_value"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

Primary tests: 13,347
BH-FDR supported tests (q < 0.05): 1,978

BH-FDR supported tests by program:
                 n_tests  n_fdr_supported
program                                  
CONSENSUS_TX_01     4449             1665
CONSENSUS_TX_02     4449               91
CONSENSUS_TX_03     4449              222

Q-value distribution:
count    1.334700e+04
mean     4.545093e-01
std      3.300835e-01
min      7.328538e-07
1%       1.541193e-03
5%       1.141562e-02
10%      2.855079e-02
25%      1.267111e-01
50%      4.380434e-01
75%      7.714773e-01
90%      9.101514e-01
95%      9.561138e-01
99%      9.902453e-01
max      9.999684e-01
Name: q_value, dtype: float64


In [32]:
# =============================================================================
# Define lineage-consistency and leave-one-project-out diagnostics
# =============================================================================

def fit_within_project_effect(
    gene: str,
    program: str,
    project_id: str,
) -> dict:
    """Estimate the within-project mutation-associated score difference."""

    project_data = primary_analysis_base.loc[
        primary_analysis_base["project_id"].eq(project_id),
        [
            "tcga_case_barcode",
            program,
        ],
    ].copy()

    project_data["mutation_status"] = (
        project_data["tcga_case_barcode"]
        .isin(mutated_cases_by_gene[gene])
        .astype(float)
    )

    design_matrix = sm.add_constant(
        project_data[["mutation_status"]],
        has_constant="add",
    )

    model = sm.OLS(
        project_data[program].astype(float),
        design_matrix,
    ).fit(
        cov_type="HC3"
    )

    confidence_interval = model.conf_int().loc[
        "mutation_status"
    ]

    return {
        "Hugo_Symbol": gene,
        "program": program,
        "project_id": project_id,
        "n_cases": len(project_data),
        "n_mutated": int(project_data["mutation_status"].sum()),
        "n_nonmutated": int(
            (project_data["mutation_status"] == 0).sum()
        ),
        "beta_mutation": float(
            model.params["mutation_status"]
        ),
        "se_hc3": float(
            model.bse["mutation_status"]
        ),
        "ci95_lower": float(
            confidence_interval.iloc[0]
        ),
        "ci95_upper": float(
            confidence_interval.iloc[1]
        ),
    }


def fit_leave_one_project_out(
    gene: str,
    program: str,
    omitted_project: str,
) -> dict:
    """Refit the primary model after omitting one supported TCGA project."""

    retained_projects = [
        project_id
        for project_id in supported_projects_by_gene[gene]
        if project_id != omitted_project
    ]

    model_data = primary_analysis_base.loc[
        primary_analysis_base["project_id"].isin(
            retained_projects
        ),
        [
            "tcga_case_barcode",
            "project_id",
            program,
        ],
    ].copy()

    model_data["mutation_status"] = (
        model_data["tcga_case_barcode"]
        .isin(mutated_cases_by_gene[gene])
        .astype(float)
    )

    project_dummies = pd.get_dummies(
        model_data["project_id"],
        prefix="project",
        drop_first=True,
        dtype=float,
    )

    design_matrix = pd.concat(
        [
            model_data[["mutation_status"]],
            project_dummies,
        ],
        axis=1,
    )

    design_matrix = sm.add_constant(
        design_matrix,
        has_constant="add",
    )

    model = sm.OLS(
        model_data[program].astype(float),
        design_matrix,
    ).fit(
        cov_type="HC3"
    )

    return {
        "Hugo_Symbol": gene,
        "program": program,
        "omitted_project": omitted_project,
        "n_projects": len(retained_projects),
        "n_cases": len(model_data),
        "beta_mutation": float(
            model.params["mutation_status"]
        ),
    }

In [33]:
# =============================================================================
# Evaluate lineage consistency and leave-one-project-out robustness
# =============================================================================

fdr_supported_pairs = (
    primary_associations.loc[
        primary_associations["primary_fdr_supported"],
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "q_value",
            "n_projects",
        ],
    ]
    .reset_index(drop=True)
)

lineage_effect_rows = []
leave_one_project_out_rows = []
recurrence_diagnostic_rows = []

n_pairs = len(fdr_supported_pairs)

for pair_index, pair in enumerate(
    fdr_supported_pairs.itertuples(index=False),
    start=1,
):
    gene = pair.Hugo_Symbol
    program = pair.program
    pooled_beta = pair.beta_mutation

    supported_projects = supported_projects_by_gene[gene]

    pair_lineage_effects = []
    pair_loo_effects = []

    for project_id in supported_projects:
        lineage_result = fit_within_project_effect(
            gene=gene,
            program=program,
            project_id=project_id,
        )
        lineage_effect_rows.append(lineage_result)
        pair_lineage_effects.append(lineage_result)

        loo_result = fit_leave_one_project_out(
            gene=gene,
            program=program,
            omitted_project=project_id,
        )
        leave_one_project_out_rows.append(loo_result)
        pair_loo_effects.append(loo_result)

    if pooled_beta > 0:
        same_direction = [
            result["beta_mutation"] > 0
            for result in pair_lineage_effects
        ]
        loo_reversal = [
            result["beta_mutation"] < 0
            for result in pair_loo_effects
        ]
    elif pooled_beta < 0:
        same_direction = [
            result["beta_mutation"] < 0
            for result in pair_lineage_effects
        ]
        loo_reversal = [
            result["beta_mutation"] > 0
            for result in pair_loo_effects
        ]
    else:
        same_direction = [
            False
            for _ in pair_lineage_effects
        ]
        loo_reversal = [
            False
            for _ in pair_loo_effects
        ]

    n_supported_projects = len(supported_projects)
    n_same_direction = int(sum(same_direction))
    directional_fraction = (
        n_same_direction / n_supported_projects
    )

    no_loo_sign_reversal = not any(loo_reversal)

    cross_cancer_recurrent = (
        pair.q_value < 0.05
        and n_supported_projects >= 3
        and n_same_direction >= 3
        and directional_fraction >= 0.70
        and no_loo_sign_reversal
    )

    recurrence_diagnostic_rows.append(
        {
            "Hugo_Symbol": gene,
            "program": program,
            "pooled_beta_mutation": pooled_beta,
            "q_value": pair.q_value,
            "n_supported_projects": n_supported_projects,
            "n_same_direction_projects": n_same_direction,
            "directional_fraction": directional_fraction,
            "no_loo_sign_reversal": no_loo_sign_reversal,
            "cross_cancer_recurrent": cross_cancer_recurrent,
        }
    )

    if pair_index % 100 == 0 or pair_index == n_pairs:
        print(
            f"Processed FDR-supported pairs: "
            f"{pair_index:,}/{n_pairs:,}"
        )

lineage_specific_effects = pd.DataFrame(
    lineage_effect_rows
)

leave_one_project_out_results = pd.DataFrame(
    leave_one_project_out_rows
)

recurrence_diagnostics = pd.DataFrame(
    recurrence_diagnostic_rows
)

assert len(recurrence_diagnostics) == n_pairs
assert recurrence_diagnostics["q_value"].lt(0.05).all()
assert recurrence_diagnostics["n_supported_projects"].ge(3).all()

print(
    "\nFDR-supported pairs evaluated:",
    f"{len(recurrence_diagnostics):,}",
)
print(
    "Lineage-specific estimates:",
    f"{len(lineage_specific_effects):,}",
)
print(
    "Leave-one-project-out estimates:",
    f"{len(leave_one_project_out_results):,}",
)
print(
    "Pairs meeting frozen cross-cancer recurrence criteria:",
    f"{int(recurrence_diagnostics['cross_cancer_recurrent'].sum()):,}",
)

print("\nDirectional-consistency summary:")
print(
    recurrence_diagnostics["directional_fraction"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nLeave-one-project-out robustness:")
print(
    recurrence_diagnostics["no_loo_sign_reversal"]
    .value_counts()
)

Processed FDR-supported pairs: 100/1,978
Processed FDR-supported pairs: 200/1,978
Processed FDR-supported pairs: 300/1,978
Processed FDR-supported pairs: 400/1,978
Processed FDR-supported pairs: 500/1,978
Processed FDR-supported pairs: 600/1,978
Processed FDR-supported pairs: 700/1,978
Processed FDR-supported pairs: 800/1,978
Processed FDR-supported pairs: 900/1,978
Processed FDR-supported pairs: 1,000/1,978
Processed FDR-supported pairs: 1,100/1,978
Processed FDR-supported pairs: 1,200/1,978
Processed FDR-supported pairs: 1,300/1,978
Processed FDR-supported pairs: 1,400/1,978
Processed FDR-supported pairs: 1,500/1,978
Processed FDR-supported pairs: 1,600/1,978
Processed FDR-supported pairs: 1,700/1,978
Processed FDR-supported pairs: 1,800/1,978
Processed FDR-supported pairs: 1,900/1,978
Processed FDR-supported pairs: 1,978/1,978

FDR-supported pairs evaluated: 1,978
Lineage-specific estimates: 11,515
Leave-one-project-out estimates: 11,515
Pairs meeting frozen cross-cancer recurrence 

In [34]:
# =============================================================================
# Integrate recurrence diagnostics and summarize primary association structure
# =============================================================================

primary_associations_annotated = (
    primary_associations
    .merge(
        recurrence_diagnostics[
            [
                "Hugo_Symbol",
                "program",
                "n_same_direction_projects",
                "directional_fraction",
                "no_loo_sign_reversal",
                "cross_cancer_recurrent",
            ]
        ],
        on=[
            "Hugo_Symbol",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

primary_associations_annotated[
    "cross_cancer_recurrent"
] = (
    primary_associations_annotated[
        "cross_cancer_recurrent"
    ]
    .fillna(False)
    .astype(bool)
)

primary_associations_annotated["effect_direction"] = (
    primary_associations_annotated["beta_mutation"]
    .gt(0)
    .map(
        {
            True: "positive",
            False: "negative",
        }
    )
)

summary_by_program = (
    primary_associations_annotated
    .groupby("program", as_index=False)
    .agg(
        n_tests=("Hugo_Symbol", "size"),
        n_fdr_supported=("primary_fdr_supported", "sum"),
        n_cross_cancer_recurrent=("cross_cancer_recurrent", "sum"),
    )
)

summary_by_program[
    "fraction_fdr_supported_recurrent"
] = (
    summary_by_program["n_cross_cancer_recurrent"]
    / summary_by_program["n_fdr_supported"]
)

recurrent_effect_summary = (
    primary_associations_annotated.loc[
        primary_associations_annotated[
            "cross_cancer_recurrent"
        ]
    ]
    .groupby("program")["beta_mutation"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
)

recurrent_direction_summary = (
    primary_associations_annotated.loc[
        primary_associations_annotated[
            "cross_cancer_recurrent"
        ]
    ]
    .groupby(
        [
            "program",
            "effect_direction",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print("Primary structure by program:")
print(summary_by_program.to_string(index=False))

print("\nCross-cancer recurrent effect-size distributions:")
print(recurrent_effect_summary)

print("\nCross-cancer recurrent effect directions:")
print(recurrent_direction_summary)

print(
    "\nTotal cross-cancer recurrent pairs:",
    f"{int(primary_associations_annotated['cross_cancer_recurrent'].sum()):,}",
)

Primary structure by program:
        program  n_tests  n_fdr_supported  n_cross_cancer_recurrent  fraction_fdr_supported_recurrent
CONSENSUS_TX_01     4449             1665                      1315                          0.789790
CONSENSUS_TX_02     4449               91                        78                          0.857143
CONSENSUS_TX_03     4449              222                       208                          0.936937

Cross-cancer recurrent effect-size distributions:
                  count      mean       std       min       10%       25%  \
program                                                                     
CONSENSUS_TX_01  1315.0  0.280071  0.079352 -0.174637  0.183652  0.224541   
CONSENSUS_TX_02    78.0 -0.079277  0.141237 -0.237832 -0.206653 -0.170606   
CONSENSUS_TX_03   208.0 -0.168389  0.065703 -0.426124 -0.236384 -0.195962   

                      50%       75%       90%       max  
program                                                  
CONSENSUS

In [35]:
# =============================================================================
# Prepare absolute-purity sensitivity cohort
# =============================================================================

purity_sensitivity_base = (
    primary_analysis_base
    .merge(
        confounder_covariates[
            [
                "case_submitter_id",
                "absolute_purity",
            ]
        ],
        on="case_submitter_id",
        how="left",
        validate="one_to_one",
    )
)

purity_complete_cases = (
    purity_sensitivity_base["absolute_purity"]
    .notna()
)

purity_sensitivity_complete = (
    purity_sensitivity_base.loc[
        purity_complete_cases
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Primary informative cases:",
    f"{len(primary_analysis_base):,}",
)
print(
    "Absolute-purity complete cases:",
    f"{len(purity_sensitivity_complete):,}",
)
print(
    "Cases excluded from purity sensitivity:",
    f"{(~purity_complete_cases).sum():,}",
)
print(
    "Complete-case fraction:",
    f"{purity_complete_cases.mean():.4f}",
)

print("\nComplete cases by project:")
print(
    purity_sensitivity_complete["project_id"]
    .value_counts()
    .sort_index()
)

assert len(purity_sensitivity_complete) == 8806
assert purity_sensitivity_complete["absolute_purity"].notna().all()

Primary informative cases: 9,134
Absolute-purity complete cases: 8,806
Cases excluded from purity sensitivity: 328
Complete-case fraction: 0.9641

Complete cases by project:
project_id
TCGA-ACC      75
TCGA-BLCA    389
TCGA-BRCA    919
TCGA-CESC    274
TCGA-CHOL     35
TCGA-COAD    410
TCGA-DLBC     36
TCGA-ESCA    161
TCGA-GBM     195
TCGA-HNSC    489
TCGA-KICH     66
TCGA-KIRC    316
TCGA-KIRP    265
TCGA-LAML     47
TCGA-LGG     501
TCGA-LIHC    351
TCGA-LUAD    490
TCGA-LUSC    474
TCGA-MESO     76
TCGA-OV      274
TCGA-PAAD    150
TCGA-PCPG    160
TCGA-PRAD    466
TCGA-READ    145
TCGA-SARC    223
TCGA-SKCM    103
TCGA-STAD    391
TCGA-TGCT    139
TCGA-THCA    454
TCGA-THYM    101
TCGA-UCEC    495
TCGA-UCS      56
TCGA-UVM      80
Name: count, dtype: int64


In [36]:
# =============================================================================
# Define absolute-purity-adjusted sensitivity model
# =============================================================================

def fit_purity_sensitivity_model(
    gene: str,
    program: str,
) -> dict:
    """Fit the prespecified purity-adjusted complete-case sensitivity model."""

    supported_projects = supported_projects_by_gene[gene]

    model_data = purity_sensitivity_complete.loc[
        purity_sensitivity_complete["project_id"].isin(
            supported_projects
        ),
        [
            "tcga_case_barcode",
            "project_id",
            "absolute_purity",
            program,
        ],
    ].copy()

    model_data["mutation_status"] = (
        model_data["tcga_case_barcode"]
        .isin(mutated_cases_by_gene[gene])
        .astype(float)
    )

    project_dummies = pd.get_dummies(
        model_data["project_id"],
        prefix="project",
        drop_first=True,
        dtype=float,
    )

    design_matrix = pd.concat(
        [
            model_data[
                [
                    "mutation_status",
                    "absolute_purity",
                ]
            ].astype(float),
            project_dummies,
        ],
        axis=1,
    )

    design_matrix = sm.add_constant(
        design_matrix,
        has_constant="add",
    )

    model = sm.OLS(
        model_data[program].astype(float),
        design_matrix,
    ).fit(
        cov_type="HC3"
    )

    confidence_interval = model.conf_int().loc[
        "mutation_status"
    ]

    project_group_support = (
        model_data
        .groupby("project_id")["mutation_status"]
        .agg(["sum", "count"])
    )

    project_group_support["n_nonmutated"] = (
        project_group_support["count"]
        - project_group_support["sum"]
    )

    n_projects_with_both_groups = int(
        (
            project_group_support["sum"].gt(0)
            & project_group_support["n_nonmutated"].gt(0)
        ).sum()
    )

    return {
        "Hugo_Symbol": gene,
        "program": program,
        "n_cases": len(model_data),
        "n_projects": model_data["project_id"].nunique(),
        "n_projects_with_both_groups": n_projects_with_both_groups,
        "n_mutated": int(model_data["mutation_status"].sum()),
        "n_nonmutated": int(
            (model_data["mutation_status"] == 0).sum()
        ),
        "beta_mutation_purity_adjusted": float(
            model.params["mutation_status"]
        ),
        "se_hc3_purity_adjusted": float(
            model.bse["mutation_status"]
        ),
        "ci95_lower_purity_adjusted": float(
            confidence_interval.iloc[0]
        ),
        "ci95_upper_purity_adjusted": float(
            confidence_interval.iloc[1]
        ),
        "p_value_purity_adjusted": float(
            model.pvalues["mutation_status"]
        ),
    }

In [37]:
# =============================================================================
# Smoke-test absolute-purity sensitivity model
# =============================================================================

purity_smoke_test_result = fit_purity_sensitivity_model(
    gene=SMOKE_TEST_GENE,
    program=SMOKE_TEST_PROGRAM,
)

assert purity_smoke_test_result["n_cases"] > 0
assert purity_smoke_test_result["n_projects"] >= PRIMARY_MIN_SUPPORTED_PROJECTS
assert purity_smoke_test_result["n_mutated"] > 0
assert purity_smoke_test_result["n_nonmutated"] > 0
assert (
    purity_smoke_test_result["n_projects_with_both_groups"]
    >= PRIMARY_MIN_SUPPORTED_PROJECTS
)

for field in [
    "beta_mutation_purity_adjusted",
    "se_hc3_purity_adjusted",
    "ci95_lower_purity_adjusted",
    "ci95_upper_purity_adjusted",
    "p_value_purity_adjusted",
]:
    assert pd.notna(purity_smoke_test_result[field])

assert purity_smoke_test_result["se_hc3_purity_adjusted"] > 0
assert (
    0.0
    <= purity_smoke_test_result["p_value_purity_adjusted"]
    <= 1.0
)

print("Purity-sensitivity smoke test passed.")
print("Gene:", SMOKE_TEST_GENE)
print("Program:", SMOKE_TEST_PROGRAM)
print("Cases:", purity_smoke_test_result["n_cases"])
print("Projects:", purity_smoke_test_result["n_projects"])
print(
    "Projects with both mutation groups:",
    purity_smoke_test_result["n_projects_with_both_groups"],
)
print("Mutated:", purity_smoke_test_result["n_mutated"])
print("Non-mutated:", purity_smoke_test_result["n_nonmutated"])

Purity-sensitivity smoke test passed.
Gene: TTN
Program: CONSENSUS_TX_01
Cases: 7935
Projects: 22
Projects with both mutation groups: 22
Mutated: 2360
Non-mutated: 5575


In [38]:
# =============================================================================
# Fit absolute-purity sensitivity for FDR-supported primary pairs
# =============================================================================

purity_sensitivity_results = []

n_pairs = len(fdr_supported_pairs)

for pair_index, pair in enumerate(
    fdr_supported_pairs.itertuples(index=False),
    start=1,
):
    purity_sensitivity_results.append(
        fit_purity_sensitivity_model(
            gene=pair.Hugo_Symbol,
            program=pair.program,
        )
    )

    if pair_index % 100 == 0 or pair_index == n_pairs:
        print(
            f"Processed purity-sensitivity pairs: "
            f"{pair_index:,}/{n_pairs:,}"
        )

purity_sensitivity_results = pd.DataFrame(
    purity_sensitivity_results
)

purity_comparison = (
    fdr_supported_pairs[
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "q_value",
        ]
    ]
    .merge(
        purity_sensitivity_results,
        on=["Hugo_Symbol", "program"],
        how="left",
        validate="one_to_one",
    )
)

purity_comparison["same_direction"] = (
    purity_comparison["beta_mutation"]
    * purity_comparison["beta_mutation_purity_adjusted"]
    > 0
)

purity_comparison["absolute_beta_change"] = (
    purity_comparison["beta_mutation_purity_adjusted"]
    - purity_comparison["beta_mutation"]
)

purity_comparison["relative_beta_magnitude"] = (
    purity_comparison["beta_mutation_purity_adjusted"].abs()
    / purity_comparison["beta_mutation"].abs()
)

assert len(purity_comparison) == n_pairs
assert purity_comparison["beta_mutation_purity_adjusted"].notna().all()

print(
    "\nPurity-sensitivity pairs fitted:",
    f"{len(purity_comparison):,}",
)
print(
    "Pairs retaining primary effect direction:",
    f"{int(purity_comparison['same_direction'].sum()):,}",
)
print(
    "Direction-retention fraction:",
    f"{purity_comparison['same_direction'].mean():.4f}",
)

print(
    "\nPairs with fewer than 3 projects retaining both mutation groups:",
    f"{int((purity_comparison['n_projects_with_both_groups'] < 3).sum()):,}",
)

print("\nRelative effect-magnitude summary:")
print(
    purity_comparison["relative_beta_magnitude"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nAbsolute beta-change summary:")
print(
    purity_comparison["absolute_beta_change"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

Processed purity-sensitivity pairs: 100/1,978
Processed purity-sensitivity pairs: 200/1,978
Processed purity-sensitivity pairs: 300/1,978
Processed purity-sensitivity pairs: 400/1,978
Processed purity-sensitivity pairs: 500/1,978
Processed purity-sensitivity pairs: 600/1,978
Processed purity-sensitivity pairs: 700/1,978
Processed purity-sensitivity pairs: 800/1,978
Processed purity-sensitivity pairs: 900/1,978
Processed purity-sensitivity pairs: 1,000/1,978
Processed purity-sensitivity pairs: 1,100/1,978
Processed purity-sensitivity pairs: 1,200/1,978
Processed purity-sensitivity pairs: 1,300/1,978
Processed purity-sensitivity pairs: 1,400/1,978
Processed purity-sensitivity pairs: 1,500/1,978
Processed purity-sensitivity pairs: 1,600/1,978
Processed purity-sensitivity pairs: 1,700/1,978
Processed purity-sensitivity pairs: 1,800/1,978
Processed purity-sensitivity pairs: 1,900/1,978
Processed purity-sensitivity pairs: 1,978/1,978

Purity-sensitivity pairs fitted: 1,978
Pairs retaining pr

In [39]:
# =============================================================================
# Prepare proliferation-score sensitivity cohort
# =============================================================================

proliferation_sensitivity_base = (
    primary_analysis_base
    .merge(
        confounder_covariates[
            [
                "case_submitter_id",
                "external_panimmune_proliferation_score",
            ]
        ],
        on="case_submitter_id",
        how="left",
        validate="one_to_one",
    )
)

proliferation_complete_cases = (
    proliferation_sensitivity_base[
        "external_panimmune_proliferation_score"
    ].notna()
)

proliferation_sensitivity_complete = (
    proliferation_sensitivity_base.loc[
        proliferation_complete_cases
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Primary informative cases:",
    f"{len(primary_analysis_base):,}",
)
print(
    "Proliferation-score complete cases:",
    f"{len(proliferation_sensitivity_complete):,}",
)
print(
    "Cases excluded from proliferation sensitivity:",
    f"{(~proliferation_complete_cases).sum():,}",
)
print(
    "Complete-case fraction:",
    f"{proliferation_complete_cases.mean():.4f}",
)

projects_with_proliferation = (
    proliferation_sensitivity_complete["project_id"]
    .nunique()
)

print(
    "Projects represented:",
    f"{projects_with_proliferation:,}",
)

print("\nComplete cases by project:")
print(
    proliferation_sensitivity_complete["project_id"]
    .value_counts()
    .sort_index()
)

assert len(proliferation_sensitivity_complete) == 8369
assert (
    proliferation_sensitivity_complete[
        "external_panimmune_proliferation_score"
    ]
    .notna()
    .all()
)

Primary informative cases: 9,134
Proliferation-score complete cases: 8,369
Cases excluded from proliferation sensitivity: 765
Complete-case fraction: 0.9162
Projects represented: 30

Complete cases by project:
project_id
TCGA-ACC      76
TCGA-BLCA    389
TCGA-BRCA    949
TCGA-CESC    280
TCGA-CHOL     34
TCGA-COAD    408
TCGA-ESCA    172
TCGA-GBM     115
TCGA-HNSC    494
TCGA-KICH     65
TCGA-KIRC    317
TCGA-KIRP    260
TCGA-LGG     504
TCGA-LIHC    353
TCGA-LUAD    444
TCGA-LUSC    467
TCGA-MESO     74
TCGA-OV      178
TCGA-PAAD    145
TCGA-PCPG    177
TCGA-PRAD    399
TCGA-READ    140
TCGA-SARC    199
TCGA-SKCM    103
TCGA-STAD    377
TCGA-TGCT    139
TCGA-THCA    481
TCGA-UCEC    493
TCGA-UCS      57
TCGA-UVM      80
Name: count, dtype: int64


In [40]:
# =============================================================================
# Define proliferation-adjusted sensitivity model
# =============================================================================

def fit_proliferation_sensitivity_model(
    gene: str,
    program: str,
) -> dict:
    """Fit the prespecified proliferation-adjusted complete-case sensitivity model."""

    supported_projects = supported_projects_by_gene[gene]

    model_data = proliferation_sensitivity_complete.loc[
        proliferation_sensitivity_complete["project_id"].isin(
            supported_projects
        ),
        [
            "tcga_case_barcode",
            "project_id",
            "external_panimmune_proliferation_score",
            program,
        ],
    ].copy()

    model_data["mutation_status"] = (
        model_data["tcga_case_barcode"]
        .isin(mutated_cases_by_gene[gene])
        .astype(float)
    )

    project_dummies = pd.get_dummies(
        model_data["project_id"],
        prefix="project",
        drop_first=True,
        dtype=float,
    )

    design_matrix = pd.concat(
        [
            model_data[
                [
                    "mutation_status",
                    "external_panimmune_proliferation_score",
                ]
            ].astype(float),
            project_dummies,
        ],
        axis=1,
    )

    design_matrix = sm.add_constant(
        design_matrix,
        has_constant="add",
    )

    model = sm.OLS(
        model_data[program].astype(float),
        design_matrix,
    ).fit(
        cov_type="HC3"
    )

    confidence_interval = model.conf_int().loc[
        "mutation_status"
    ]

    project_group_support = (
        model_data
        .groupby("project_id")["mutation_status"]
        .agg(["sum", "count"])
    )

    project_group_support["n_nonmutated"] = (
        project_group_support["count"]
        - project_group_support["sum"]
    )

    n_projects_with_both_groups = int(
        (
            project_group_support["sum"].gt(0)
            & project_group_support["n_nonmutated"].gt(0)
        ).sum()
    )

    return {
        "Hugo_Symbol": gene,
        "program": program,
        "n_cases": len(model_data),
        "n_projects": model_data["project_id"].nunique(),
        "n_projects_with_both_groups": n_projects_with_both_groups,
        "n_mutated": int(model_data["mutation_status"].sum()),
        "n_nonmutated": int(
            (model_data["mutation_status"] == 0).sum()
        ),
        "beta_mutation_proliferation_adjusted": float(
            model.params["mutation_status"]
        ),
        "se_hc3_proliferation_adjusted": float(
            model.bse["mutation_status"]
        ),
        "ci95_lower_proliferation_adjusted": float(
            confidence_interval.iloc[0]
        ),
        "ci95_upper_proliferation_adjusted": float(
            confidence_interval.iloc[1]
        ),
        "p_value_proliferation_adjusted": float(
            model.pvalues["mutation_status"]
        ),
    }

In [41]:
# =============================================================================
# Smoke-test proliferation sensitivity model
# =============================================================================

proliferation_smoke_test_result = (
    fit_proliferation_sensitivity_model(
        gene=SMOKE_TEST_GENE,
        program=SMOKE_TEST_PROGRAM,
    )
)

assert proliferation_smoke_test_result["n_cases"] > 0
assert (
    proliferation_smoke_test_result["n_projects"]
    >= PRIMARY_MIN_SUPPORTED_PROJECTS
)
assert proliferation_smoke_test_result["n_mutated"] > 0
assert proliferation_smoke_test_result["n_nonmutated"] > 0

for field in [
    "beta_mutation_proliferation_adjusted",
    "se_hc3_proliferation_adjusted",
    "ci95_lower_proliferation_adjusted",
    "ci95_upper_proliferation_adjusted",
    "p_value_proliferation_adjusted",
]:
    assert pd.notna(
        proliferation_smoke_test_result[field]
    )

assert (
    proliferation_smoke_test_result[
        "se_hc3_proliferation_adjusted"
    ]
    > 0
)

assert (
    0.0
    <= proliferation_smoke_test_result[
        "p_value_proliferation_adjusted"
    ]
    <= 1.0
)

print("Proliferation-sensitivity smoke test passed.")
print("Gene:", SMOKE_TEST_GENE)
print("Program:", SMOKE_TEST_PROGRAM)
print(
    "Cases:",
    proliferation_smoke_test_result["n_cases"],
)
print(
    "Projects:",
    proliferation_smoke_test_result["n_projects"],
)
print(
    "Projects with both mutation groups:",
    proliferation_smoke_test_result[
        "n_projects_with_both_groups"
    ],
)
print(
    "Mutated:",
    proliferation_smoke_test_result["n_mutated"],
)
print(
    "Non-mutated:",
    proliferation_smoke_test_result["n_nonmutated"],
)

Proliferation-sensitivity smoke test passed.
Gene: TTN
Program: CONSENSUS_TX_01
Cases: 7667
Projects: 22
Projects with both mutation groups: 22
Mutated: 2273
Non-mutated: 5394


In [42]:
# =============================================================================
# Fit proliferation sensitivity for FDR-supported primary pairs
# =============================================================================

proliferation_sensitivity_results = []

n_pairs = len(fdr_supported_pairs)

for pair_index, pair in enumerate(
    fdr_supported_pairs.itertuples(index=False),
    start=1,
):
    proliferation_sensitivity_results.append(
        fit_proliferation_sensitivity_model(
            gene=pair.Hugo_Symbol,
            program=pair.program,
        )
    )

    if pair_index % 100 == 0 or pair_index == n_pairs:
        print(
            f"Processed proliferation-sensitivity pairs: "
            f"{pair_index:,}/{n_pairs:,}"
        )

proliferation_sensitivity_results = pd.DataFrame(
    proliferation_sensitivity_results
)

proliferation_comparison = (
    fdr_supported_pairs[
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "q_value",
        ]
    ]
    .merge(
        proliferation_sensitivity_results,
        on=["Hugo_Symbol", "program"],
        how="left",
        validate="one_to_one",
    )
)

proliferation_comparison["same_direction"] = (
    proliferation_comparison["beta_mutation"]
    * proliferation_comparison[
        "beta_mutation_proliferation_adjusted"
    ]
    > 0
)

proliferation_comparison["absolute_beta_change"] = (
    proliferation_comparison[
        "beta_mutation_proliferation_adjusted"
    ]
    - proliferation_comparison["beta_mutation"]
)

proliferation_comparison["relative_beta_magnitude"] = (
    proliferation_comparison[
        "beta_mutation_proliferation_adjusted"
    ].abs()
    / proliferation_comparison["beta_mutation"].abs()
)

assert len(proliferation_comparison) == n_pairs
assert (
    proliferation_comparison[
        "beta_mutation_proliferation_adjusted"
    ]
    .notna()
    .all()
)

print(
    "\nProliferation-sensitivity pairs fitted:",
    f"{len(proliferation_comparison):,}",
)
print(
    "Pairs retaining primary effect direction:",
    f"{int(proliferation_comparison['same_direction'].sum()):,}",
)
print(
    "Direction-retention fraction:",
    f"{proliferation_comparison['same_direction'].mean():.4f}",
)

print(
    "\nPairs with fewer than 3 projects retaining both mutation groups:",
    f"{int((proliferation_comparison['n_projects_with_both_groups'] < 3).sum()):,}",
)

print("\nRelative effect-magnitude summary:")
print(
    proliferation_comparison["relative_beta_magnitude"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nAbsolute beta-change summary:")
print(
    proliferation_comparison["absolute_beta_change"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

Processed proliferation-sensitivity pairs: 100/1,978
Processed proliferation-sensitivity pairs: 200/1,978
Processed proliferation-sensitivity pairs: 300/1,978
Processed proliferation-sensitivity pairs: 400/1,978
Processed proliferation-sensitivity pairs: 500/1,978
Processed proliferation-sensitivity pairs: 600/1,978
Processed proliferation-sensitivity pairs: 700/1,978
Processed proliferation-sensitivity pairs: 800/1,978
Processed proliferation-sensitivity pairs: 900/1,978
Processed proliferation-sensitivity pairs: 1,000/1,978
Processed proliferation-sensitivity pairs: 1,100/1,978
Processed proliferation-sensitivity pairs: 1,200/1,978
Processed proliferation-sensitivity pairs: 1,300/1,978
Processed proliferation-sensitivity pairs: 1,400/1,978
Processed proliferation-sensitivity pairs: 1,500/1,978
Processed proliferation-sensitivity pairs: 1,600/1,978
Processed proliferation-sensitivity pairs: 1,700/1,978
Processed proliferation-sensitivity pairs: 1,800/1,978
Processed proliferation-sens

In [43]:
# =============================================================================
# Prepare single-MAF sensitivity cohort
# =============================================================================

single_maf_cases = (
    case_payload_status.loc[
        (case_payload_status["n_mafs"].astype("int64") == 1)
        & (~case_payload_status["all_mafs_empty"]),
        "tcga_case_barcode",
    ]
    .drop_duplicates()
)

single_maf_sensitivity_base = (
    primary_analysis_base.loc[
        primary_analysis_base["tcga_case_barcode"].isin(
            single_maf_cases
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Primary informative cases:",
    f"{len(primary_analysis_base):,}",
)
print(
    "Single-MAF informative cases:",
    f"{len(single_maf_sensitivity_base):,}",
)
print(
    "Cases excluded from single-MAF sensitivity:",
    f"{len(primary_analysis_base) - len(single_maf_sensitivity_base):,}",
)
print(
    "Single-MAF fraction:",
    f"{len(single_maf_sensitivity_base) / len(primary_analysis_base):.4f}",
)
print(
    "Projects represented:",
    f"{single_maf_sensitivity_base['project_id'].nunique():,}",
)

print("\nSingle-MAF informative cases by project:")
print(
    single_maf_sensitivity_base["project_id"]
    .value_counts()
    .sort_index()
)

assert len(single_maf_sensitivity_base) == 8920

Primary informative cases: 9,134
Single-MAF informative cases: 8,920
Cases excluded from single-MAF sensitivity: 214
Single-MAF fraction: 0.9766
Projects represented: 33

Single-MAF informative cases by project:
project_id
TCGA-ACC      77
TCGA-BLCA    398
TCGA-BRCA    954
TCGA-CESC    284
TCGA-CHOL     35
TCGA-COAD    407
TCGA-DLBC     36
TCGA-ESCA    183
TCGA-GBM     144
TCGA-HNSC    500
TCGA-KICH     66
TCGA-KIRC    314
TCGA-KIRP    267
TCGA-LAML     55
TCGA-LGG     506
TCGA-LIHC    362
TCGA-LUAD    484
TCGA-LUSC    432
TCGA-MESO     78
TCGA-OV      241
TCGA-PAAD    165
TCGA-PCPG    178
TCGA-PRAD    489
TCGA-READ    143
TCGA-SARC    232
TCGA-SKCM    103
TCGA-STAD    401
TCGA-TGCT    140
TCGA-THCA    486
TCGA-THYM    117
TCGA-UCEC    506
TCGA-UCS      57
TCGA-UVM      80
Name: count, dtype: int64


In [44]:
# =============================================================================
# Define single-MAF sensitivity model
# =============================================================================

def fit_single_maf_sensitivity_model(
    gene: str,
    program: str,
) -> dict:
    """Fit the prespecified single-MAF sensitivity model."""

    supported_projects = supported_projects_by_gene[gene]

    model_data = single_maf_sensitivity_base.loc[
        single_maf_sensitivity_base["project_id"].isin(
            supported_projects
        ),
        [
            "tcga_case_barcode",
            "project_id",
            program,
        ],
    ].copy()

    model_data["mutation_status"] = (
        model_data["tcga_case_barcode"]
        .isin(mutated_cases_by_gene[gene])
        .astype(float)
    )

    project_dummies = pd.get_dummies(
        model_data["project_id"],
        prefix="project",
        drop_first=True,
        dtype=float,
    )

    design_matrix = pd.concat(
        [
            model_data[["mutation_status"]],
            project_dummies,
        ],
        axis=1,
    )

    design_matrix = sm.add_constant(
        design_matrix,
        has_constant="add",
    )

    model = sm.OLS(
        model_data[program].astype(float),
        design_matrix,
    ).fit(
        cov_type="HC3"
    )

    confidence_interval = model.conf_int().loc[
        "mutation_status"
    ]

    project_group_support = (
        model_data
        .groupby("project_id")["mutation_status"]
        .agg(["sum", "count"])
    )

    project_group_support["n_nonmutated"] = (
        project_group_support["count"]
        - project_group_support["sum"]
    )

    n_projects_with_both_groups = int(
        (
            project_group_support["sum"].gt(0)
            & project_group_support["n_nonmutated"].gt(0)
        ).sum()
    )

    return {
        "Hugo_Symbol": gene,
        "program": program,
        "n_cases": len(model_data),
        "n_projects": model_data["project_id"].nunique(),
        "n_projects_with_both_groups": n_projects_with_both_groups,
        "n_mutated": int(model_data["mutation_status"].sum()),
        "n_nonmutated": int(
            (model_data["mutation_status"] == 0).sum()
        ),
        "beta_mutation_single_maf": float(
            model.params["mutation_status"]
        ),
        "se_hc3_single_maf": float(
            model.bse["mutation_status"]
        ),
        "ci95_lower_single_maf": float(
            confidence_interval.iloc[0]
        ),
        "ci95_upper_single_maf": float(
            confidence_interval.iloc[1]
        ),
        "p_value_single_maf": float(
            model.pvalues["mutation_status"]
        ),
    }

In [45]:
# =============================================================================
# Smoke-test single-MAF sensitivity model
# =============================================================================

single_maf_smoke_test_result = (
    fit_single_maf_sensitivity_model(
        gene=SMOKE_TEST_GENE,
        program=SMOKE_TEST_PROGRAM,
    )
)

assert single_maf_smoke_test_result["n_cases"] > 0
assert (
    single_maf_smoke_test_result["n_projects"]
    >= PRIMARY_MIN_SUPPORTED_PROJECTS
)
assert single_maf_smoke_test_result["n_mutated"] > 0
assert single_maf_smoke_test_result["n_nonmutated"] > 0

for field in [
    "beta_mutation_single_maf",
    "se_hc3_single_maf",
    "ci95_lower_single_maf",
    "ci95_upper_single_maf",
    "p_value_single_maf",
]:
    assert pd.notna(
        single_maf_smoke_test_result[field]
    )

assert single_maf_smoke_test_result["se_hc3_single_maf"] > 0

assert (
    0.0
    <= single_maf_smoke_test_result["p_value_single_maf"]
    <= 1.0
)

print("Single-MAF sensitivity smoke test passed.")
print("Gene:", SMOKE_TEST_GENE)
print("Program:", SMOKE_TEST_PROGRAM)
print(
    "Cases:",
    single_maf_smoke_test_result["n_cases"],
)
print(
    "Projects:",
    single_maf_smoke_test_result["n_projects"],
)
print(
    "Projects with both mutation groups:",
    single_maf_smoke_test_result[
        "n_projects_with_both_groups"
    ],
)
print(
    "Mutated:",
    single_maf_smoke_test_result["n_mutated"],
)
print(
    "Non-mutated:",
    single_maf_smoke_test_result["n_nonmutated"],
)

Single-MAF sensitivity smoke test passed.
Gene: TTN
Program: CONSENSUS_TX_01
Cases: 8001
Projects: 22
Projects with both mutation groups: 22
Mutated: 2336
Non-mutated: 5665


In [46]:
# =============================================================================
# Fit single-MAF sensitivity for FDR-supported primary pairs
# =============================================================================

single_maf_sensitivity_results = []

n_pairs = len(fdr_supported_pairs)

for pair_index, pair in enumerate(
    fdr_supported_pairs.itertuples(index=False),
    start=1,
):
    single_maf_sensitivity_results.append(
        fit_single_maf_sensitivity_model(
            gene=pair.Hugo_Symbol,
            program=pair.program,
        )
    )

    if pair_index % 100 == 0 or pair_index == n_pairs:
        print(
            f"Processed single-MAF sensitivity pairs: "
            f"{pair_index:,}/{n_pairs:,}"
        )

single_maf_sensitivity_results = pd.DataFrame(
    single_maf_sensitivity_results
)

single_maf_comparison = (
    fdr_supported_pairs[
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "q_value",
        ]
    ]
    .merge(
        single_maf_sensitivity_results,
        on=["Hugo_Symbol", "program"],
        how="left",
        validate="one_to_one",
    )
)

single_maf_comparison["same_direction"] = (
    single_maf_comparison["beta_mutation"]
    * single_maf_comparison["beta_mutation_single_maf"]
    > 0
)

single_maf_comparison["absolute_beta_change"] = (
    single_maf_comparison["beta_mutation_single_maf"]
    - single_maf_comparison["beta_mutation"]
)

single_maf_comparison["relative_beta_magnitude"] = (
    single_maf_comparison["beta_mutation_single_maf"].abs()
    / single_maf_comparison["beta_mutation"].abs()
)

assert len(single_maf_comparison) == n_pairs
assert (
    single_maf_comparison["beta_mutation_single_maf"]
    .notna()
    .all()
)

print(
    "\nSingle-MAF sensitivity pairs fitted:",
    f"{len(single_maf_comparison):,}",
)
print(
    "Pairs retaining primary effect direction:",
    f"{int(single_maf_comparison['same_direction'].sum()):,}",
)
print(
    "Direction-retention fraction:",
    f"{single_maf_comparison['same_direction'].mean():.4f}",
)

print(
    "\nPairs with fewer than 3 projects retaining both mutation groups:",
    f"{int((single_maf_comparison['n_projects_with_both_groups'] < 3).sum()):,}",
)

print("\nRelative effect-magnitude summary:")
print(
    single_maf_comparison["relative_beta_magnitude"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nAbsolute beta-change summary:")
print(
    single_maf_comparison["absolute_beta_change"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

Processed single-MAF sensitivity pairs: 100/1,978
Processed single-MAF sensitivity pairs: 200/1,978
Processed single-MAF sensitivity pairs: 300/1,978
Processed single-MAF sensitivity pairs: 400/1,978
Processed single-MAF sensitivity pairs: 500/1,978
Processed single-MAF sensitivity pairs: 600/1,978
Processed single-MAF sensitivity pairs: 700/1,978
Processed single-MAF sensitivity pairs: 800/1,978
Processed single-MAF sensitivity pairs: 900/1,978
Processed single-MAF sensitivity pairs: 1,000/1,978
Processed single-MAF sensitivity pairs: 1,100/1,978
Processed single-MAF sensitivity pairs: 1,200/1,978
Processed single-MAF sensitivity pairs: 1,300/1,978
Processed single-MAF sensitivity pairs: 1,400/1,978
Processed single-MAF sensitivity pairs: 1,500/1,978
Processed single-MAF sensitivity pairs: 1,600/1,978
Processed single-MAF sensitivity pairs: 1,700/1,978
Processed single-MAF sensitivity pairs: 1,800/1,978
Processed single-MAF sensitivity pairs: 1,900/1,978
Processed single-MAF sensitivi

In [47]:
# =============================================================================
# Construct extended-splice sensitivity mutation representation
# =============================================================================

EXTENDED_SPLICE_CLASSIFICATION = "Splice_Region"

splice_region_frames = []
raw_splice_region_rows = 0

for row in maf_file_table.itertuples(index=False):
    maf = pd.read_csv(
        row.maf_path,
        sep="\t",
        comment="#",
        usecols=QUALIFYING_MAF_FIELDS,
        dtype="string",
        low_memory=False,
    )

    if maf.empty:
        continue

    splice_region = maf.loc[
        maf["Variant_Classification"].eq(
            EXTENDED_SPLICE_CLASSIFICATION
        )
    ].copy()

    if splice_region.empty:
        continue

    raw_splice_region_rows += len(splice_region)

    splice_region["tcga_case_barcode"] = row.tcga_case_barcode
    splice_region["project_id"] = row.project_id
    splice_region["file_id"] = row.file_id

    splice_region_frames.append(splice_region)

splice_region_variants_raw = pd.concat(
    splice_region_frames,
    ignore_index=True,
)

splice_region_variants = (
    splice_region_variants_raw
    .drop_duplicates(
        subset=CASE_VARIANT_KEY,
        keep="first",
    )
    .reset_index(drop=True)
)

extended_qualifying_variants = (
    pd.concat(
        [
            qualifying_variants,
            splice_region_variants,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=CASE_VARIANT_KEY,
        keep="first",
    )
    .reset_index(drop=True)
)

extended_case_gene_observed = (
    extended_qualifying_variants[
        [
            "tcga_case_barcode",
            "project_id",
            "Hugo_Symbol",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

additional_case_gene_observations = (
    len(extended_case_gene_observed)
    - len(case_gene_observed)
)

print(
    "Raw Splice_Region rows:",
    f"{raw_splice_region_rows:,}",
)
print(
    "Unique case-level Splice_Region variants:",
    f"{len(splice_region_variants):,}",
)
print(
    "Extended qualifying variants:",
    f"{len(extended_qualifying_variants):,}",
)
print(
    "Extended case-gene observations:",
    f"{len(extended_case_gene_observed):,}",
)
print(
    "Additional case-gene observations vs primary:",
    f"{additional_case_gene_observations:,}",
)
print(
    "Genes represented in extended definition:",
    f"{extended_case_gene_observed['Hugo_Symbol'].nunique():,}",
)

missing_extended_gene = (
    extended_qualifying_variants["Hugo_Symbol"].isna()
    | extended_qualifying_variants["Hugo_Symbol"].str.strip().eq("")
)

print(
    "Extended qualifying variants with missing/blank Hugo_Symbol:",
    f"{int(missing_extended_gene.sum()):,}",
)

Raw Splice_Region rows: 13,689
Unique case-level Splice_Region variants: 13,256
Extended qualifying variants: 1,570,502
Extended case-gene observations: 1,363,739
Additional case-gene observations vs primary: 10,554
Genes represented in extended definition: 18,947
Extended qualifying variants with missing/blank Hugo_Symbol: 0


In [48]:
# =============================================================================
# Prepare extended-splice mutation indices for frozen primary gene universe
# =============================================================================

extended_mutated_cases_by_gene = (
    extended_case_gene_observed.loc[
        extended_case_gene_observed["Hugo_Symbol"].isin(
            primary_gene_universe["Hugo_Symbol"]
        )
    ]
    .groupby("Hugo_Symbol")["tcga_case_barcode"]
    .apply(set)
    .to_dict()
)

missing_primary_genes = (
    set(primary_gene_universe["Hugo_Symbol"])
    - set(extended_mutated_cases_by_gene)
)

extended_gene_analysis_sizes = []

for gene in primary_gene_universe["Hugo_Symbol"]:
    supported_projects = supported_projects_by_gene[gene]

    gene_cases = primary_analysis_base.loc[
        primary_analysis_base["project_id"].isin(
            supported_projects
        ),
        [
            "tcga_case_barcode",
            "project_id",
        ],
    ].copy()

    gene_cases["mutation_status"] = (
        gene_cases["tcga_case_barcode"]
        .isin(extended_mutated_cases_by_gene[gene])
        .astype("int8")
    )

    extended_gene_analysis_sizes.append(
        {
            "Hugo_Symbol": gene,
            "n_projects": gene_cases["project_id"].nunique(),
            "n_cases": len(gene_cases),
            "n_mutated": int(gene_cases["mutation_status"].sum()),
            "n_nonmutated": int(
                (gene_cases["mutation_status"] == 0).sum()
            ),
        }
    )

extended_gene_analysis_sizes = pd.DataFrame(
    extended_gene_analysis_sizes
)

print(
    "Primary genes represented under extended definition:",
    f"{len(extended_mutated_cases_by_gene):,}",
)
print(
    "Primary genes missing under extended definition:",
    f"{len(missing_primary_genes):,}",
)

print("\nChange in mutated-case count vs primary:")
mutation_count_change = (
    extended_gene_analysis_sizes
    .set_index("Hugo_Symbol")["n_mutated"]
    - gene_analysis_sizes
    .set_index("Hugo_Symbol")["n_mutated"]
)

print(
    mutation_count_change.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print(
    "\nGenes with at least one additional mutated case:",
    f"{int(mutation_count_change.gt(0).sum()):,}",
)

assert len(missing_primary_genes) == 0
assert mutation_count_change.ge(0).all()

Primary genes represented under extended definition: 4,449
Primary genes missing under extended definition: 0

Change in mutated-case count vs primary:
count    4449.000000
mean        0.623960
std         1.203065
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
90%         2.000000
95%         3.000000
99%         5.000000
max        34.000000
Name: n_mutated, dtype: float64

Genes with at least one additional mutated case: 1,666


In [49]:
# =============================================================================
# Define extended-splice sensitivity model
# =============================================================================

def fit_extended_splice_sensitivity_model(
    gene: str,
    program: str,
) -> dict:
    """Fit the prespecified extended-splice sensitivity model."""

    supported_projects = supported_projects_by_gene[gene]

    model_data = primary_analysis_base.loc[
        primary_analysis_base["project_id"].isin(
            supported_projects
        ),
        [
            "tcga_case_barcode",
            "project_id",
            program,
        ],
    ].copy()

    model_data["mutation_status"] = (
        model_data["tcga_case_barcode"]
        .isin(extended_mutated_cases_by_gene[gene])
        .astype(float)
    )

    project_dummies = pd.get_dummies(
        model_data["project_id"],
        prefix="project",
        drop_first=True,
        dtype=float,
    )

    design_matrix = pd.concat(
        [
            model_data[["mutation_status"]],
            project_dummies,
        ],
        axis=1,
    )

    design_matrix = sm.add_constant(
        design_matrix,
        has_constant="add",
    )

    model = sm.OLS(
        model_data[program].astype(float),
        design_matrix,
    ).fit(
        cov_type="HC3"
    )

    confidence_interval = model.conf_int().loc[
        "mutation_status"
    ]

    project_group_support = (
        model_data
        .groupby("project_id")["mutation_status"]
        .agg(["sum", "count"])
    )

    project_group_support["n_nonmutated"] = (
        project_group_support["count"]
        - project_group_support["sum"]
    )

    n_projects_with_both_groups = int(
        (
            project_group_support["sum"].gt(0)
            & project_group_support["n_nonmutated"].gt(0)
        ).sum()
    )

    return {
        "Hugo_Symbol": gene,
        "program": program,
        "n_cases": len(model_data),
        "n_projects": model_data["project_id"].nunique(),
        "n_projects_with_both_groups": n_projects_with_both_groups,
        "n_mutated": int(model_data["mutation_status"].sum()),
        "n_nonmutated": int(
            (model_data["mutation_status"] == 0).sum()
        ),
        "beta_mutation_extended_splice": float(
            model.params["mutation_status"]
        ),
        "se_hc3_extended_splice": float(
            model.bse["mutation_status"]
        ),
        "ci95_lower_extended_splice": float(
            confidence_interval.iloc[0]
        ),
        "ci95_upper_extended_splice": float(
            confidence_interval.iloc[1]
        ),
        "p_value_extended_splice": float(
            model.pvalues["mutation_status"]
        ),
    }

In [50]:
# =============================================================================
# Smoke-test extended-splice sensitivity model
# =============================================================================

extended_splice_smoke_test_result = (
    fit_extended_splice_sensitivity_model(
        gene=SMOKE_TEST_GENE,
        program=SMOKE_TEST_PROGRAM,
    )
)

assert extended_splice_smoke_test_result["n_cases"] > 0
assert (
    extended_splice_smoke_test_result["n_projects"]
    >= PRIMARY_MIN_SUPPORTED_PROJECTS
)
assert extended_splice_smoke_test_result["n_mutated"] > 0
assert extended_splice_smoke_test_result["n_nonmutated"] > 0
assert (
    extended_splice_smoke_test_result[
        "n_projects_with_both_groups"
    ]
    >= PRIMARY_MIN_SUPPORTED_PROJECTS
)

for field in [
    "beta_mutation_extended_splice",
    "se_hc3_extended_splice",
    "ci95_lower_extended_splice",
    "ci95_upper_extended_splice",
    "p_value_extended_splice",
]:
    assert pd.notna(
        extended_splice_smoke_test_result[field]
    )

assert (
    extended_splice_smoke_test_result[
        "se_hc3_extended_splice"
    ]
    > 0
)

assert (
    0.0
    <= extended_splice_smoke_test_result[
        "p_value_extended_splice"
    ]
    <= 1.0
)

print("Extended-splice sensitivity smoke test passed.")
print("Gene:", SMOKE_TEST_GENE)
print("Program:", SMOKE_TEST_PROGRAM)
print(
    "Cases:",
    extended_splice_smoke_test_result["n_cases"],
)
print(
    "Projects:",
    extended_splice_smoke_test_result["n_projects"],
)
print(
    "Projects with both mutation groups:",
    extended_splice_smoke_test_result[
        "n_projects_with_both_groups"
    ],
)
print(
    "Mutated:",
    extended_splice_smoke_test_result["n_mutated"],
)
print(
    "Non-mutated:",
    extended_splice_smoke_test_result["n_nonmutated"],
)

Extended-splice sensitivity smoke test passed.
Gene: TTN
Program: CONSENSUS_TX_01
Cases: 8212
Projects: 22
Projects with both mutation groups: 22
Mutated: 2433
Non-mutated: 5779


In [51]:
# =============================================================================
# Fit extended-splice sensitivity for FDR-supported primary pairs
# =============================================================================

extended_splice_sensitivity_results = []

n_pairs = len(fdr_supported_pairs)

for pair_index, pair in enumerate(
    fdr_supported_pairs.itertuples(index=False),
    start=1,
):
    extended_splice_sensitivity_results.append(
        fit_extended_splice_sensitivity_model(
            gene=pair.Hugo_Symbol,
            program=pair.program,
        )
    )

    if pair_index % 100 == 0 or pair_index == n_pairs:
        print(
            f"Processed extended-splice sensitivity pairs: "
            f"{pair_index:,}/{n_pairs:,}"
        )

extended_splice_sensitivity_results = pd.DataFrame(
    extended_splice_sensitivity_results
)

extended_splice_comparison = (
    fdr_supported_pairs[
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "q_value",
        ]
    ]
    .merge(
        extended_splice_sensitivity_results,
        on=["Hugo_Symbol", "program"],
        how="left",
        validate="one_to_one",
    )
)

extended_splice_comparison["same_direction"] = (
    extended_splice_comparison["beta_mutation"]
    * extended_splice_comparison[
        "beta_mutation_extended_splice"
    ]
    > 0
)

extended_splice_comparison["absolute_beta_change"] = (
    extended_splice_comparison[
        "beta_mutation_extended_splice"
    ]
    - extended_splice_comparison["beta_mutation"]
)

extended_splice_comparison["relative_beta_magnitude"] = (
    extended_splice_comparison[
        "beta_mutation_extended_splice"
    ].abs()
    / extended_splice_comparison["beta_mutation"].abs()
)

assert len(extended_splice_comparison) == n_pairs
assert (
    extended_splice_comparison[
        "beta_mutation_extended_splice"
    ]
    .notna()
    .all()
)

print(
    "\nExtended-splice sensitivity pairs fitted:",
    f"{len(extended_splice_comparison):,}",
)
print(
    "Pairs retaining primary effect direction:",
    f"{int(extended_splice_comparison['same_direction'].sum()):,}",
)
print(
    "Direction-retention fraction:",
    f"{extended_splice_comparison['same_direction'].mean():.4f}",
)

print(
    "\nPairs with fewer than 3 projects retaining both mutation groups:",
    f"{int((extended_splice_comparison['n_projects_with_both_groups'] < 3).sum()):,}",
)

print("\nRelative effect-magnitude summary:")
print(
    extended_splice_comparison["relative_beta_magnitude"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nAbsolute beta-change summary:")
print(
    extended_splice_comparison["absolute_beta_change"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

Processed extended-splice sensitivity pairs: 100/1,978
Processed extended-splice sensitivity pairs: 200/1,978
Processed extended-splice sensitivity pairs: 300/1,978
Processed extended-splice sensitivity pairs: 400/1,978
Processed extended-splice sensitivity pairs: 500/1,978
Processed extended-splice sensitivity pairs: 600/1,978
Processed extended-splice sensitivity pairs: 700/1,978
Processed extended-splice sensitivity pairs: 800/1,978
Processed extended-splice sensitivity pairs: 900/1,978
Processed extended-splice sensitivity pairs: 1,000/1,978
Processed extended-splice sensitivity pairs: 1,100/1,978
Processed extended-splice sensitivity pairs: 1,200/1,978
Processed extended-splice sensitivity pairs: 1,300/1,978
Processed extended-splice sensitivity pairs: 1,400/1,978
Processed extended-splice sensitivity pairs: 1,500/1,978
Processed extended-splice sensitivity pairs: 1,600/1,978
Processed extended-splice sensitivity pairs: 1,700/1,978
Processed extended-splice sensitivity pairs: 1,80

In [52]:
# =============================================================================
# Consolidate prespecified sensitivity diagnostics
# =============================================================================

sensitivity_stability = (
    fdr_supported_pairs[
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "q_value",
        ]
    ]
    .merge(
        purity_comparison[
            [
                "Hugo_Symbol",
                "program",
                "same_direction",
                "relative_beta_magnitude",
                "n_projects_with_both_groups",
            ]
        ].rename(
            columns={
                "same_direction": "purity_same_direction",
                "relative_beta_magnitude": "purity_relative_beta",
                "n_projects_with_both_groups": "purity_supported_projects",
            }
        ),
        on=["Hugo_Symbol", "program"],
        validate="one_to_one",
    )
    .merge(
        proliferation_comparison[
            [
                "Hugo_Symbol",
                "program",
                "same_direction",
                "relative_beta_magnitude",
                "n_projects_with_both_groups",
            ]
        ].rename(
            columns={
                "same_direction": "proliferation_same_direction",
                "relative_beta_magnitude": "proliferation_relative_beta",
                "n_projects_with_both_groups": "proliferation_supported_projects",
            }
        ),
        on=["Hugo_Symbol", "program"],
        validate="one_to_one",
    )
    .merge(
        single_maf_comparison[
            [
                "Hugo_Symbol",
                "program",
                "same_direction",
                "relative_beta_magnitude",
                "n_projects_with_both_groups",
            ]
        ].rename(
            columns={
                "same_direction": "single_maf_same_direction",
                "relative_beta_magnitude": "single_maf_relative_beta",
                "n_projects_with_both_groups": "single_maf_supported_projects",
            }
        ),
        on=["Hugo_Symbol", "program"],
        validate="one_to_one",
    )
    .merge(
        extended_splice_comparison[
            [
                "Hugo_Symbol",
                "program",
                "same_direction",
                "relative_beta_magnitude",
                "n_projects_with_both_groups",
            ]
        ].rename(
            columns={
                "same_direction": "extended_splice_same_direction",
                "relative_beta_magnitude": "extended_splice_relative_beta",
                "n_projects_with_both_groups": "extended_splice_supported_projects",
            }
        ),
        on=["Hugo_Symbol", "program"],
        validate="one_to_one",
    )
)

direction_columns = [
    "purity_same_direction",
    "proliferation_same_direction",
    "single_maf_same_direction",
    "extended_splice_same_direction",
]

support_columns = [
    "purity_supported_projects",
    "proliferation_supported_projects",
    "single_maf_supported_projects",
    "extended_splice_supported_projects",
]

sensitivity_stability["all_sensitivities_same_direction"] = (
    sensitivity_stability[direction_columns].all(axis=1)
)

sensitivity_stability["all_sensitivities_lineage_supported"] = (
    sensitivity_stability[support_columns]
    .ge(PRIMARY_MIN_SUPPORTED_PROJECTS)
    .all(axis=1)
)

sensitivity_stability["sensitivity_stable"] = (
    sensitivity_stability["all_sensitivities_same_direction"]
    & sensitivity_stability["all_sensitivities_lineage_supported"]
)

assert len(sensitivity_stability) == len(fdr_supported_pairs)

print(
    "FDR-supported pairs:",
    f"{len(sensitivity_stability):,}",
)
print(
    "Same direction in all four sensitivities:",
    f"{int(sensitivity_stability['all_sensitivities_same_direction'].sum()):,}",
)
print(
    "Adequate lineage support in all four sensitivities:",
    f"{int(sensitivity_stability['all_sensitivities_lineage_supported'].sum()):,}",
)
print(
    "Sensitivity-stable pairs:",
    f"{int(sensitivity_stability['sensitivity_stable'].sum()):,}",
)

print("\nSensitivity stability by program:")
print(
    sensitivity_stability
    .groupby("program")["sensitivity_stable"]
    .agg(
        n_fdr_supported="size",
        n_sensitivity_stable="sum",
        fraction_sensitivity_stable="mean",
    )
)

FDR-supported pairs: 1,978
Same direction in all four sensitivities: 1,977
Adequate lineage support in all four sensitivities: 1,975
Sensitivity-stable pairs: 1,975

Sensitivity stability by program:
                 n_fdr_supported  n_sensitivity_stable  \
program                                                  
CONSENSUS_TX_01             1665                  1664   
CONSENSUS_TX_02               91                    90   
CONSENSUS_TX_03              222                   221   

                 fraction_sensitivity_stable  
program                                       
CONSENSUS_TX_01                     0.999399  
CONSENSUS_TX_02                     0.989011  
CONSENSUS_TX_03                     0.995495  


In [53]:
# =============================================================================
# Construct descriptive observed qualifying-variant burden
# =============================================================================

case_qualifying_variant_burden = (
    qualifying_variants
    .groupby(
        [
            "tcga_case_barcode",
            "project_id",
        ],
        as_index=False,
    )
    .agg(
        n_observed_qualifying_variants=(
            "Tumor_Seq_Allele2",
            "size",
        ),
    )
)

case_qualifying_variant_burden = (
    informative_case_table
    .merge(
        case_qualifying_variant_burden,
        on=[
            "tcga_case_barcode",
            "project_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

case_qualifying_variant_burden[
    "n_observed_qualifying_variants"
] = (
    case_qualifying_variant_burden[
        "n_observed_qualifying_variants"
    ]
    .fillna(0)
    .astype("int64")
)

case_qualifying_variant_burden["log1p_observed_qualifying_variants"] = (
    case_qualifying_variant_burden[
        "n_observed_qualifying_variants"
    ]
    .map(lambda value: float(__import__("math").log1p(value)))
)

print(
    "Informative cases:",
    f"{len(case_qualifying_variant_burden):,}",
)
print(
    "Cases with zero qualifying variants:",
    f"{int((case_qualifying_variant_burden['n_observed_qualifying_variants'] == 0).sum()):,}",
)

print("\nObserved qualifying-variant count distribution:")
print(
    case_qualifying_variant_burden[
        "n_observed_qualifying_variants"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

Informative cases: 9,134
Cases with zero qualifying variants: 13

Observed qualifying-variant count distribution:
count     9134.000000
mean       170.488942
std        715.855077
min          0.000000
25%         22.000000
50%         49.000000
75%        106.000000
90%        274.700000
95%        534.000000
99%       1835.450000
max      22911.000000
Name: n_observed_qualifying_variants, dtype: float64


In [54]:
# =============================================================================
# Diagnose association between observed variant burden and program scores
# =============================================================================

burden_program_data = (
    primary_analysis_base
    .merge(
        case_qualifying_variant_burden[
            [
                "tcga_case_barcode",
                "log1p_observed_qualifying_variants",
            ]
        ],
        on="tcga_case_barcode",
        how="left",
        validate="one_to_one",
    )
)

burden_program_results = []

for program in PROGRAM_COLUMNS:
    model_data = burden_program_data[
        [
            "project_id",
            "log1p_observed_qualifying_variants",
            program,
        ]
    ].copy()

    project_dummies = pd.get_dummies(
        model_data["project_id"],
        prefix="project",
        drop_first=True,
        dtype=float,
    )

    design_matrix = pd.concat(
        [
            model_data[
                ["log1p_observed_qualifying_variants"]
            ].astype(float),
            project_dummies,
        ],
        axis=1,
    )

    design_matrix = sm.add_constant(
        design_matrix,
        has_constant="add",
    )

    model = sm.OLS(
        model_data[program].astype(float),
        design_matrix,
    ).fit(
        cov_type="HC3"
    )

    confidence_interval = model.conf_int().loc[
        "log1p_observed_qualifying_variants"
    ]

    burden_program_results.append(
        {
            "program": program,
            "n_cases": len(model_data),
            "beta_log1p_variant_burden": float(
                model.params[
                    "log1p_observed_qualifying_variants"
                ]
            ),
            "se_hc3": float(
                model.bse[
                    "log1p_observed_qualifying_variants"
                ]
            ),
            "ci95_lower": float(
                confidence_interval.iloc[0]
            ),
            "ci95_upper": float(
                confidence_interval.iloc[1]
            ),
            "p_value": float(
                model.pvalues[
                    "log1p_observed_qualifying_variants"
                ]
            ),
        }
    )

burden_program_results = pd.DataFrame(
    burden_program_results
)

print(
    burden_program_results.to_string(
        index=False
    )
)

        program  n_cases  beta_log1p_variant_burden   se_hc3  ci95_lower  ci95_upper      p_value
CONSENSUS_TX_01     9134                   0.031659 0.010224    0.011620    0.051697 1.957768e-03
CONSENSUS_TX_02     9134                  -0.028774 0.006318   -0.041157   -0.016390 5.259826e-06
CONSENSUS_TX_03     9134                  -0.103597 0.008616   -0.120484   -0.086710 2.658903e-33


In [55]:
# =============================================================================
# Diagnose gene mutation status versus background observed variant burden
# =============================================================================

fdr_supported_genes = sorted(
    fdr_supported_pairs["Hugo_Symbol"].unique()
)

focal_gene_variant_counts = (
    qualifying_variants.loc[
        qualifying_variants["Hugo_Symbol"].isin(
            fdr_supported_genes
        )
    ]
    .groupby(
        [
            "Hugo_Symbol",
            "tcga_case_barcode",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "n_focal_gene_variants"})
)

background_burden_base = (
    primary_analysis_base[
        [
            "tcga_case_barcode",
            "project_id",
        ]
    ]
    .merge(
        case_qualifying_variant_burden[
            [
                "tcga_case_barcode",
                "n_observed_qualifying_variants",
            ]
        ],
        on="tcga_case_barcode",
        how="left",
        validate="one_to_one",
    )
)

background_burden_results = []

for gene_index, gene in enumerate(
    fdr_supported_genes,
    start=1,
):
    supported_projects = supported_projects_by_gene[gene]

    model_data = background_burden_base.loc[
        background_burden_base["project_id"].isin(
            supported_projects
        )
    ].copy()

    gene_counts = (
        focal_gene_variant_counts.loc[
            focal_gene_variant_counts["Hugo_Symbol"].eq(gene),
            [
                "tcga_case_barcode",
                "n_focal_gene_variants",
            ],
        ]
        .set_index("tcga_case_barcode")[
            "n_focal_gene_variants"
        ]
    )

    model_data["n_focal_gene_variants"] = (
        model_data["tcga_case_barcode"]
        .map(gene_counts)
        .fillna(0)
        .astype("int64")
    )

    model_data["background_variant_count"] = (
        model_data["n_observed_qualifying_variants"]
        - model_data["n_focal_gene_variants"]
    )

    assert model_data["background_variant_count"].ge(0).all()

    model_data["log1p_background_variant_burden"] = (
        model_data["background_variant_count"]
        .map(lambda value: float(__import__("math").log1p(value)))
    )

    model_data["mutation_status"] = (
        model_data["tcga_case_barcode"]
        .isin(mutated_cases_by_gene[gene])
        .astype(float)
    )

    project_dummies = pd.get_dummies(
        model_data["project_id"],
        prefix="project",
        drop_first=True,
        dtype=float,
    )

    design_matrix = pd.concat(
        [
            model_data[["mutation_status"]],
            project_dummies,
        ],
        axis=1,
    )

    design_matrix = sm.add_constant(
        design_matrix,
        has_constant="add",
    )

    model = sm.OLS(
        model_data["log1p_background_variant_burden"],
        design_matrix,
    ).fit(
        cov_type="HC3"
    )

    background_burden_results.append(
        {
            "Hugo_Symbol": gene,
            "n_cases": len(model_data),
            "n_projects": model_data["project_id"].nunique(),
            "beta_background_burden": float(
                model.params["mutation_status"]
            ),
            "se_hc3": float(
                model.bse["mutation_status"]
            ),
            "p_value": float(
                model.pvalues["mutation_status"]
            ),
        }
    )

    if gene_index % 250 == 0 or gene_index == len(fdr_supported_genes):
        print(
            f"Processed FDR-supported genes: "
            f"{gene_index:,}/{len(fdr_supported_genes):,}"
        )

background_burden_results = pd.DataFrame(
    background_burden_results
)

print(
    "\nFDR-supported unique genes:",
    f"{len(background_burden_results):,}",
)

print("\nBackground-burden coefficient distribution:")
print(
    background_burden_results["beta_background_burden"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
)

print(
    "\nGenes with positive background-burden coefficient:",
    f"{int(background_burden_results['beta_background_burden'].gt(0).sum()):,}",
)
print(
    "Genes with negative background-burden coefficient:",
    f"{int(background_burden_results['beta_background_burden'].lt(0).sum()):,}",
)

Processed FDR-supported genes: 250/1,797
Processed FDR-supported genes: 500/1,797
Processed FDR-supported genes: 750/1,797
Processed FDR-supported genes: 1,000/1,797
Processed FDR-supported genes: 1,250/1,797
Processed FDR-supported genes: 1,500/1,797
Processed FDR-supported genes: 1,750/1,797
Processed FDR-supported genes: 1,797/1,797

FDR-supported unique genes: 1,797

Background-burden coefficient distribution:
count    1797.000000
mean        2.111180
std         0.452109
min         0.029877
10%         1.558120
25%         1.787844
50%         2.116025
75%         2.444110
90%         2.689924
max         3.235895
Name: beta_background_burden, dtype: float64

Genes with positive background-burden coefficient: 1,797
Genes with negative background-burden coefficient: 0


In [56]:
# =============================================================================
# Quantify alignment of gene-program effects with background mutation burden
# =============================================================================

from scipy.stats import pearsonr, spearmanr


burden_program_direction = (
    burden_program_results
    .set_index("program")["beta_log1p_variant_burden"]
    .to_dict()
)

burden_alignment = (
    fdr_supported_pairs[
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "q_value",
        ]
    ]
    .merge(
        background_burden_results[
            [
                "Hugo_Symbol",
                "beta_background_burden",
            ]
        ],
        on="Hugo_Symbol",
        how="left",
        validate="many_to_one",
    )
)

burden_alignment["expected_burden_direction"] = (
    burden_alignment["program"]
    .map(burden_program_direction)
)

burden_alignment["effect_matches_burden_direction"] = (
    burden_alignment["beta_mutation"]
    * burden_alignment["expected_burden_direction"]
    > 0
)

correlation_rows = []

for program, frame in burden_alignment.groupby(
    "program",
    sort=True,
):
    pearson_r, pearson_p = pearsonr(
        frame["beta_background_burden"],
        frame["beta_mutation"],
    )

    spearman_rho, spearman_p = spearmanr(
        frame["beta_background_burden"],
        frame["beta_mutation"],
    )

    correlation_rows.append(
        {
            "program": program,
            "n_pairs": len(frame),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
            "fraction_matching_burden_direction": (
                frame["effect_matches_burden_direction"].mean()
            ),
        }
    )

burden_alignment_summary = pd.DataFrame(
    correlation_rows
)

print("Gene-program alignment with background mutation burden:")
print(
    burden_alignment_summary.to_string(
        index=False
    )
)

print("\nDirection concordance counts:")
print(
    burden_alignment
    .groupby("program")[
        "effect_matches_burden_direction"
    ]
    .value_counts()
    .unstack(fill_value=0)
)

Gene-program alignment with background mutation burden:
        program  n_pairs  pearson_r     pearson_p  spearman_rho    spearman_p  fraction_matching_burden_direction
CONSENSUS_TX_01     1665   0.691556 3.319723e-237      0.718464 1.744643e-264                            0.998198
CONSENSUS_TX_02       91  -0.264386  1.132694e-02     -0.510925  2.283316e-07                            0.736264
CONSENSUS_TX_03      222  -0.496978  2.994864e-15     -0.657409  7.454880e-29                            0.986486

Direction concordance counts:
effect_matches_burden_direction  False  True 
program                                      
CONSENSUS_TX_01                      3   1662
CONSENSUS_TX_02                     24     67
CONSENSUS_TX_03                      3    219


In [57]:
# =============================================================================
# Define exploratory background-burden-conditioned model
# =============================================================================

def fit_background_burden_conditioned_model(
    gene: str,
    program: str,
) -> dict:
    """Fit exploratory model conditioned on focal-gene-excluded variant burden."""

    supported_projects = supported_projects_by_gene[gene]

    model_data = primary_analysis_base.loc[
        primary_analysis_base["project_id"].isin(
            supported_projects
        ),
        [
            "tcga_case_barcode",
            "project_id",
            program,
        ],
    ].copy()

    gene_variant_counts = (
        qualifying_variants.loc[
            qualifying_variants["Hugo_Symbol"].eq(gene),
            [
                "tcga_case_barcode",
            ],
        ]
        .groupby("tcga_case_barcode")
        .size()
    )

    total_variant_counts = (
        case_qualifying_variant_burden
        .set_index("tcga_case_barcode")[
            "n_observed_qualifying_variants"
        ]
    )

    model_data["n_total_qualifying_variants"] = (
        model_data["tcga_case_barcode"]
        .map(total_variant_counts)
        .astype("int64")
    )

    model_data["n_focal_gene_variants"] = (
        model_data["tcga_case_barcode"]
        .map(gene_variant_counts)
        .fillna(0)
        .astype("int64")
    )

    model_data["background_variant_count"] = (
        model_data["n_total_qualifying_variants"]
        - model_data["n_focal_gene_variants"]
    )

    assert model_data["background_variant_count"].ge(0).all()

    model_data["log1p_background_variant_burden"] = (
        model_data["background_variant_count"]
        .map(lambda value: float(__import__("math").log1p(value)))
    )

    model_data["mutation_status"] = (
        model_data["tcga_case_barcode"]
        .isin(mutated_cases_by_gene[gene])
        .astype(float)
    )

    project_dummies = pd.get_dummies(
        model_data["project_id"],
        prefix="project",
        drop_first=True,
        dtype=float,
    )

    design_matrix = pd.concat(
        [
            model_data[
                [
                    "mutation_status",
                    "log1p_background_variant_burden",
                ]
            ].astype(float),
            project_dummies,
        ],
        axis=1,
    )

    design_matrix = sm.add_constant(
        design_matrix,
        has_constant="add",
    )

    model = sm.OLS(
        model_data[program].astype(float),
        design_matrix,
    ).fit(
        cov_type="HC3"
    )

    confidence_interval = model.conf_int().loc[
        "mutation_status"
    ]

    return {
        "Hugo_Symbol": gene,
        "program": program,
        "n_cases": len(model_data),
        "n_projects": model_data["project_id"].nunique(),
        "beta_mutation_burden_conditioned": float(
            model.params["mutation_status"]
        ),
        "se_hc3_burden_conditioned": float(
            model.bse["mutation_status"]
        ),
        "ci95_lower_burden_conditioned": float(
            confidence_interval.iloc[0]
        ),
        "ci95_upper_burden_conditioned": float(
            confidence_interval.iloc[1]
        ),
        "p_value_burden_conditioned": float(
            model.pvalues["mutation_status"]
        ),
    }

In [58]:
# =============================================================================
# Smoke-test exploratory background-burden-conditioned model
# =============================================================================

background_burden_smoke_test_result = (
    fit_background_burden_conditioned_model(
        gene=SMOKE_TEST_GENE,
        program=SMOKE_TEST_PROGRAM,
    )
)

assert background_burden_smoke_test_result["n_cases"] > 0
assert (
    background_burden_smoke_test_result["n_projects"]
    >= PRIMARY_MIN_SUPPORTED_PROJECTS
)

for field in [
    "beta_mutation_burden_conditioned",
    "se_hc3_burden_conditioned",
    "ci95_lower_burden_conditioned",
    "ci95_upper_burden_conditioned",
    "p_value_burden_conditioned",
]:
    assert pd.notna(
        background_burden_smoke_test_result[field]
    )

assert (
    background_burden_smoke_test_result[
        "se_hc3_burden_conditioned"
    ]
    > 0
)

assert (
    0.0
    <= background_burden_smoke_test_result[
        "p_value_burden_conditioned"
    ]
    <= 1.0
)

print("Background-burden-conditioned smoke test passed.")
print("Gene:", SMOKE_TEST_GENE)
print("Program:", SMOKE_TEST_PROGRAM)
print(
    "Cases:",
    background_burden_smoke_test_result["n_cases"],
)
print(
    "Projects:",
    background_burden_smoke_test_result["n_projects"],
)

Background-burden-conditioned smoke test passed.
Gene: TTN
Program: CONSENSUS_TX_01
Cases: 8212
Projects: 22


In [59]:
# =============================================================================
# Fit exploratory background-burden-conditioned models
# =============================================================================

background_burden_conditioned_results = []

n_pairs = len(fdr_supported_pairs)

for pair_index, pair in enumerate(
    fdr_supported_pairs.itertuples(index=False),
    start=1,
):
    background_burden_conditioned_results.append(
        fit_background_burden_conditioned_model(
            gene=pair.Hugo_Symbol,
            program=pair.program,
        )
    )

    if pair_index % 100 == 0 or pair_index == n_pairs:
        print(
            f"Processed background-burden-conditioned pairs: "
            f"{pair_index:,}/{n_pairs:,}"
        )

background_burden_conditioned_results = pd.DataFrame(
    background_burden_conditioned_results
)

background_burden_conditioned_comparison = (
    fdr_supported_pairs[
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "q_value",
        ]
    ]
    .merge(
        background_burden_conditioned_results,
        on=["Hugo_Symbol", "program"],
        how="left",
        validate="one_to_one",
    )
)

background_burden_conditioned_comparison["same_direction"] = (
    background_burden_conditioned_comparison["beta_mutation"]
    * background_burden_conditioned_comparison[
        "beta_mutation_burden_conditioned"
    ]
    > 0
)

background_burden_conditioned_comparison[
    "relative_beta_magnitude"
] = (
    background_burden_conditioned_comparison[
        "beta_mutation_burden_conditioned"
    ].abs()
    / background_burden_conditioned_comparison[
        "beta_mutation"
    ].abs()
)

background_burden_conditioned_comparison[
    "absolute_beta_change"
] = (
    background_burden_conditioned_comparison[
        "beta_mutation_burden_conditioned"
    ]
    - background_burden_conditioned_comparison[
        "beta_mutation"
    ]
)

assert len(
    background_burden_conditioned_comparison
) == n_pairs

assert (
    background_burden_conditioned_comparison[
        "beta_mutation_burden_conditioned"
    ]
    .notna()
    .all()
)

print(
    "\nBackground-burden-conditioned pairs fitted:",
    f"{len(background_burden_conditioned_comparison):,}",
)

print(
    "Pairs retaining primary effect direction:",
    f"{int(background_burden_conditioned_comparison['same_direction'].sum()):,}",
)

print(
    "Direction-retention fraction:",
    f"{background_burden_conditioned_comparison['same_direction'].mean():.4f}",
)

print("\nDirection retention by program:")
print(
    background_burden_conditioned_comparison
    .groupby("program")["same_direction"]
    .agg(
        n_pairs="size",
        n_same_direction="sum",
        fraction_same_direction="mean",
    )
)

print("\nRelative effect-magnitude summary by program:")
print(
    background_burden_conditioned_comparison
    .groupby("program")["relative_beta_magnitude"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nAbsolute beta-change summary by program:")
print(
    background_burden_conditioned_comparison
    .groupby("program")["absolute_beta_change"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

Processed background-burden-conditioned pairs: 100/1,978
Processed background-burden-conditioned pairs: 200/1,978
Processed background-burden-conditioned pairs: 300/1,978
Processed background-burden-conditioned pairs: 400/1,978
Processed background-burden-conditioned pairs: 500/1,978
Processed background-burden-conditioned pairs: 600/1,978
Processed background-burden-conditioned pairs: 700/1,978
Processed background-burden-conditioned pairs: 800/1,978
Processed background-burden-conditioned pairs: 900/1,978
Processed background-burden-conditioned pairs: 1,000/1,978
Processed background-burden-conditioned pairs: 1,100/1,978
Processed background-burden-conditioned pairs: 1,200/1,978
Processed background-burden-conditioned pairs: 1,300/1,978
Processed background-burden-conditioned pairs: 1,400/1,978
Processed background-burden-conditioned pairs: 1,500/1,978
Processed background-burden-conditioned pairs: 1,600/1,978
Processed background-burden-conditioned pairs: 1,700/1,978
Processed backg

In [60]:
# =============================================================================
# Consolidate primary, recurrence, sensitivity, and burden diagnostics
# =============================================================================

final_gene_program_summary = (
    primary_associations_annotated
    .merge(
        sensitivity_stability[
            [
                "Hugo_Symbol",
                "program",
                "all_sensitivities_same_direction",
                "all_sensitivities_lineage_supported",
                "sensitivity_stable",
                "purity_relative_beta",
                "proliferation_relative_beta",
                "single_maf_relative_beta",
                "extended_splice_relative_beta",
            ]
        ],
        on=["Hugo_Symbol", "program"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        background_burden_conditioned_comparison[
            [
                "Hugo_Symbol",
                "program",
                "beta_mutation_burden_conditioned",
                "same_direction",
                "relative_beta_magnitude",
                "absolute_beta_change",
            ]
        ].rename(
            columns={
                "same_direction": "burden_conditioned_same_direction",
                "relative_beta_magnitude": "burden_conditioned_relative_beta",
                "absolute_beta_change": "burden_conditioned_beta_change",
            }
        ),
        on=["Hugo_Symbol", "program"],
        how="left",
        validate="one_to_one",
    )
)

fdr_mask = final_gene_program_summary["primary_fdr_supported"]

assert len(final_gene_program_summary) == len(primary_associations)
assert (
    final_gene_program_summary.loc[
        fdr_mask,
        "sensitivity_stable",
    ]
    .notna()
    .all()
)
assert (
    final_gene_program_summary.loc[
        fdr_mask,
        "beta_mutation_burden_conditioned",
    ]
    .notna()
    .all()
)

print(
    "Complete primary gene-program tests:",
    f"{len(final_gene_program_summary):,}",
)
print(
    "Primary FDR-supported:",
    f"{int(fdr_mask.sum()):,}",
)
print(
    "Cross-cancer recurrent:",
    f"{int(final_gene_program_summary['cross_cancer_recurrent'].sum()):,}",
)
print(
    "Sensitivity-stable:",
    f"{int(final_gene_program_summary['sensitivity_stable'].fillna(False).sum()):,}",
)

print("\nBurden-conditioned diagnostics among FDR-supported pairs:")
print(
    final_gene_program_summary.loc[fdr_mask]
    .groupby("program")
    .agg(
        n_pairs=("Hugo_Symbol", "size"),
        n_same_direction=(
            "burden_conditioned_same_direction",
            "sum",
        ),
        median_relative_beta=(
            "burden_conditioned_relative_beta",
            "median",
        ),
        mean_relative_beta=(
            "burden_conditioned_relative_beta",
            "mean",
        ),
    )
)

Complete primary gene-program tests: 13,347
Primary FDR-supported: 1,978
Cross-cancer recurrent: 1,601
Sensitivity-stable: 1,975

Burden-conditioned diagnostics among FDR-supported pairs:
                 n_pairs n_same_direction  median_relative_beta  \
program                                                           
CONSENSUS_TX_01     1665             1658              0.494515   
CONSENSUS_TX_02       91               87              0.522574   
CONSENSUS_TX_03      222               87              0.253799   

                 mean_relative_beta  
program                              
CONSENSUS_TX_01            0.487523  
CONSENSUS_TX_02            0.653644  
CONSENSUS_TX_03            0.324845  


In [61]:
# =============================================================================
# Characterize burden sensitivity within cross-cancer recurrent associations
# =============================================================================

recurrent_burden_summary = (
    final_gene_program_summary.loc[
        final_gene_program_summary["cross_cancer_recurrent"]
    ]
    .copy()
)

print(
    "Cross-cancer recurrent pairs:",
    f"{len(recurrent_burden_summary):,}",
)

print("\nRecurrent pairs by program:")
print(
    recurrent_burden_summary["program"]
    .value_counts()
    .sort_index()
)

print("\nBurden-conditioned direction retention:")
print(
    recurrent_burden_summary
    .groupby("program")
    .agg(
        n_recurrent=("Hugo_Symbol", "size"),
        n_same_direction=(
            "burden_conditioned_same_direction",
            "sum",
        ),
        fraction_same_direction=(
            "burden_conditioned_same_direction",
            "mean",
        ),
    )
)

print("\nBurden-conditioned relative effect magnitude:")
print(
    recurrent_burden_summary
    .groupby("program")[
        "burden_conditioned_relative_beta"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
)

print("\nPrimary effect magnitude among recurrent pairs:")
print(
    recurrent_burden_summary
    .groupby("program")["beta_mutation"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
)

Cross-cancer recurrent pairs:

 1,601

Recurrent pairs by program:
program
CONSENSUS_TX_01    1315
CONSENSUS_TX_02      78
CONSENSUS_TX_03     208
Name: count, dtype: int64

Burden-conditioned direction retention:
                 n_recurrent n_same_direction fraction_same_direction
program                                                              
CONSENSUS_TX_01         1315             1310                0.996198
CONSENSUS_TX_02           78               75                0.961538
CONSENSUS_TX_03          208               81                0.389423

Burden-conditioned relative effect magnitude:
                  count      mean       std       min       10%       25%  \
program                                                                     
CONSENSUS_TX_01  1315.0  0.500506  0.154775  0.019921  0.302625  0.403066   
CONSENSUS_TX_02    78.0  0.608665  0.416172  0.023911  0.183192  0.299784   
CONSENSUS_TX_03   208.0  0.315327  0.250812  0.001410  0.054073  0.131828   

                      50%       75

In [62]:
# =============================================================================
# Inspect recurrent associations with complete diagnostic context
# =============================================================================

recurrent_review_table = (
    final_gene_program_summary.loc[
        final_gene_program_summary["cross_cancer_recurrent"]
    ]
    [
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "ci95_lower",
            "ci95_upper",
            "q_value",
            "n_projects",
            "n_same_direction_projects",
            "directional_fraction",
            "no_loo_sign_reversal",
            "sensitivity_stable",
            "purity_relative_beta",
            "proliferation_relative_beta",
            "single_maf_relative_beta",
            "extended_splice_relative_beta",
            "beta_mutation_burden_conditioned",
            "burden_conditioned_same_direction",
            "burden_conditioned_relative_beta",
        ]
    ]
    .copy()
)

recurrent_review_table["absolute_primary_beta"] = (
    recurrent_review_table["beta_mutation"].abs()
)

recurrent_review_table = (
    recurrent_review_table
    .sort_values(
        [
            "program",
            "absolute_primary_beta",
            "Hugo_Symbol",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

for program in PROGRAM_COLUMNS:
    print(f"\n{program} — largest absolute primary effects")
    print(
        recurrent_review_table.loc[
            recurrent_review_table["program"].eq(program),
            [
                "Hugo_Symbol",
                "beta_mutation",
                "q_value",
                "n_projects",
                "directional_fraction",
                "sensitivity_stable",
                "beta_mutation_burden_conditioned",
                "burden_conditioned_same_direction",
                "burden_conditioned_relative_beta",
            ],
        ]
        .head(12)
        .to_string(index=False)
    )


CONSENSUS_TX_01 — largest absolute primary effects
  Hugo_Symbol  beta_mutation  q_value  n_projects  directional_fraction sensitivity_stable  beta_mutation_burden_conditioned burden_conditioned_same_direction  burden_conditioned_relative_beta
         TGM4       0.556502 0.000341           3                   1.0               True                          0.404093                              True                          0.726130
       SH3RF2       0.553376 0.000923           3                   1.0               True                          0.399409                              True                          0.721768
       YTHDF2       0.527893 0.000574           3                   1.0               True                          0.342893                              True                          0.649551
      PPP2R3B       0.525197 0.003192           3                   1.0               True                          0.365039                              True                  

In [63]:
# =============================================================================
# Characterize cross-program overlap among recurrent gene associations
# =============================================================================

recurrent_gene_program = (
    recurrent_review_table[
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "q_value",
            "burden_conditioned_relative_beta",
        ]
    ]
    .copy()
)

recurrent_program_matrix = (
    recurrent_gene_program
    .assign(recurrent=True)
    .pivot(
        index="Hugo_Symbol",
        columns="program",
        values="recurrent",
    )
    .fillna(False)
    .astype(bool)
)

for program in PROGRAM_COLUMNS:
    if program not in recurrent_program_matrix.columns:
        recurrent_program_matrix[program] = False

recurrent_program_matrix = (
    recurrent_program_matrix[PROGRAM_COLUMNS]
)

recurrent_program_matrix["n_recurrent_programs"] = (
    recurrent_program_matrix[PROGRAM_COLUMNS]
    .sum(axis=1)
)

print(
    "Unique genes with >=1 recurrent association:",
    f"{len(recurrent_program_matrix):,}",
)

print("\nNumber of recurrent programs per gene:")
print(
    recurrent_program_matrix["n_recurrent_programs"]
    .value_counts()
    .sort_index()
)

print("\nPairwise recurrent-gene overlap:")
for i, program_a in enumerate(PROGRAM_COLUMNS):
    for program_b in PROGRAM_COLUMNS[i + 1:]:
        overlap = int(
            (
                recurrent_program_matrix[program_a]
                & recurrent_program_matrix[program_b]
            ).sum()
        )

        union = int(
            (
                recurrent_program_matrix[program_a]
                | recurrent_program_matrix[program_b]
            ).sum()
        )

        jaccard = overlap / union if union else float("nan")

        print(
            f"{program_a} vs {program_b}: "
            f"overlap={overlap:,}, "
            f"Jaccard={jaccard:.4f}"
        )

print(
    "\nGenes recurrent for all three programs:",
    f"{int(recurrent_program_matrix['n_recurrent_programs'].eq(3).sum()):,}",
)

Unique genes with >=1 recurrent association: 1,476

Number of recurrent programs per gene:
n_recurrent_programs
1    1355
2     117
3       4
Name: count, dtype: int64

Pairwise recurrent-gene overlap:
CONSENSUS_TX_01 vs CONSENSUS_TX_02: overlap=30, Jaccard=0.0220
CONSENSUS_TX_01 vs CONSENSUS_TX_03: overlap=86, Jaccard=0.0598
CONSENSUS_TX_02 vs CONSENSUS_TX_03: overlap=13, Jaccard=0.0476

Genes recurrent for all three programs: 4


In [64]:
# =============================================================================
# Inspect genes recurrent across multiple consensus programs
# =============================================================================

multi_program_genes = (
    recurrent_program_matrix.loc[
        recurrent_program_matrix["n_recurrent_programs"] >= 2
    ]
    .index
)

multi_program_recurrent = (
    recurrent_review_table.loc[
        recurrent_review_table["Hugo_Symbol"].isin(
            multi_program_genes
        ),
        [
            "Hugo_Symbol",
            "program",
            "beta_mutation",
            "q_value",
            "n_projects",
            "directional_fraction",
            "beta_mutation_burden_conditioned",
            "burden_conditioned_same_direction",
            "burden_conditioned_relative_beta",
        ],
    ]
    .sort_values(
        [
            "Hugo_Symbol",
            "program",
        ]
    )
    .reset_index(drop=True)
)

multi_program_gene_summary = (
    multi_program_recurrent
    .groupby("Hugo_Symbol", as_index=False)
    .agg(
        n_recurrent_programs=("program", "nunique"),
        min_directional_fraction=(
            "directional_fraction",
            "min",
        ),
        all_burden_conditioned_same_direction=(
            "burden_conditioned_same_direction",
            "all",
        ),
        min_burden_conditioned_relative_beta=(
            "burden_conditioned_relative_beta",
            "min",
        ),
    )
)

print(
    "Genes recurrent in >=2 programs:",
    f"{len(multi_program_gene_summary):,}",
)

print("\nMulti-program gene counts:")
print(
    multi_program_gene_summary[
        "n_recurrent_programs"
    ]
    .value_counts()
    .sort_index()
)

print("\nGenes recurrent across all three programs:")
print(
    multi_program_recurrent.loc[
        multi_program_recurrent["Hugo_Symbol"].isin(
            multi_program_gene_summary.loc[
                multi_program_gene_summary[
                    "n_recurrent_programs"
                ].eq(3),
                "Hugo_Symbol",
            ]
        )
    ].to_string(index=False)
)

print(
    "\nMulti-program genes retaining direction "
    "after burden conditioning in every recurrent program:",
    f"{int(multi_program_gene_summary['all_burden_conditioned_same_direction'].sum()):,}",
)

Genes recurrent in >=2 programs: 121

Multi-program gene counts:
n_recurrent_programs
2    117
3      4
Name: count, dtype: int64

Genes recurrent across all three programs:
Hugo_Symbol         program  beta_mutation  q_value  n_projects  directional_fraction  beta_mutation_burden_conditioned burden_conditioned_same_direction  burden_conditioned_relative_beta
       KLC2 CONSENSUS_TX_01       0.396974 0.005239           3              1.000000                          0.217862                              True                          0.548806
       KLC2 CONSENSUS_TX_02      -0.209432 0.023749           3              1.000000                         -0.071494                              True                          0.341371
       KLC2 CONSENSUS_TX_03      -0.426124 0.000056           3              1.000000                         -0.294926                              True                          0.692112
      NPHP4 CONSENSUS_TX_01       0.286575 0.016575           5           

In [65]:
# =============================================================================
# Characterize program-exclusive versus shared recurrent associations
# =============================================================================

recurrent_gene_membership = (
    recurrent_program_matrix[
        ["n_recurrent_programs"]
    ]
    .reset_index()
)

recurrent_partition = (
    recurrent_review_table
    .merge(
        recurrent_gene_membership,
        on="Hugo_Symbol",
        how="left",
        validate="many_to_one",
    )
)

recurrent_partition["recurrent_scope"] = (
    recurrent_partition["n_recurrent_programs"]
    .eq(1)
    .map(
        {
            True: "program_exclusive",
            False: "shared_across_programs",
        }
    )
)

partition_summary = (
    recurrent_partition
    .groupby(
        [
            "program",
            "recurrent_scope",
        ],
        as_index=False,
    )
    .agg(
        n_pairs=("Hugo_Symbol", "size"),
        n_burden_direction_retained=(
            "burden_conditioned_same_direction",
            "sum",
        ),
        median_primary_beta=(
            "beta_mutation",
            "median",
        ),
        median_burden_conditioned_relative_beta=(
            "burden_conditioned_relative_beta",
            "median",
        ),
    )
)

partition_summary[
    "fraction_burden_direction_retained"
] = (
    partition_summary[
        "n_burden_direction_retained"
    ]
    / partition_summary["n_pairs"]
)

print("Recurrent association partition:")
print(
    partition_summary.to_string(
        index=False
    )
)

print("\nProgram-exclusive recurrent genes:")
for program in PROGRAM_COLUMNS:
    n_unique = recurrent_partition.loc[
        recurrent_partition["program"].eq(program)
        & recurrent_partition["recurrent_scope"].eq(
            "program_exclusive"
        ),
        "Hugo_Symbol",
    ].nunique()

    print(
        f"{program}: {n_unique:,}"
    )

print(
    "\nShared recurrent genes:",
    f"{int(recurrent_gene_membership['n_recurrent_programs'].ge(2).sum()):,}",
)

Recurrent association partition:
        program        recurrent_scope  n_pairs n_burden_direction_retained  median_primary_beta  median_burden_conditioned_relative_beta fraction_burden_direction_retained
CONSENSUS_TX_01      program_exclusive     1203                        1198             0.280502                                 0.504996                           0.995844
CONSENSUS_TX_01 shared_across_programs      112                         112             0.206139                                 0.520929                                1.0
CONSENSUS_TX_02      program_exclusive       39                          39            -0.105410                                 0.502343                                1.0
CONSENSUS_TX_02 shared_across_programs       39                          36            -0.149805                                 0.493170                           0.923077
CONSENSUS_TX_03      program_exclusive      113                          45            -0.158340      

In [66]:
# =============================================================================
# Characterize lineage breadth of recurrent associations
# =============================================================================

recurrent_lineage_breadth = (
    recurrent_partition[
        [
            "Hugo_Symbol",
            "program",
            "recurrent_scope",
            "n_projects",
            "directional_fraction",
            "beta_mutation",
            "burden_conditioned_relative_beta",
        ]
    ]
    .copy()
)

breadth_summary = (
    recurrent_lineage_breadth
    .groupby(
        [
            "program",
            "recurrent_scope",
        ],
        as_index=False,
    )
    .agg(
        n_pairs=("Hugo_Symbol", "size"),
        median_supported_projects=("n_projects", "median"),
        min_supported_projects=("n_projects", "min"),
        max_supported_projects=("n_projects", "max"),
        n_exactly_3_projects=(
            "n_projects",
            lambda values: int(values.eq(3).sum()),
        ),
        n_at_least_5_projects=(
            "n_projects",
            lambda values: int(values.ge(5).sum()),
        ),
        n_at_least_10_projects=(
            "n_projects",
            lambda values: int(values.ge(10).sum()),
        ),
    )
)

breadth_summary["fraction_exactly_3_projects"] = (
    breadth_summary["n_exactly_3_projects"]
    / breadth_summary["n_pairs"]
)

breadth_summary["fraction_at_least_5_projects"] = (
    breadth_summary["n_at_least_5_projects"]
    / breadth_summary["n_pairs"]
)

breadth_summary["fraction_at_least_10_projects"] = (
    breadth_summary["n_at_least_10_projects"]
    / breadth_summary["n_pairs"]
)

print("Lineage breadth of recurrent associations:")
print(
    breadth_summary.to_string(
        index=False
    )
)

print("\nSupported-project distribution by program:")
print(
    recurrent_lineage_breadth
    .groupby("program")["n_projects"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
        ]
    )
)

print("\nRecurrent pairs by supported-project count:")
print(
    recurrent_lineage_breadth
    .groupby(
        [
            "program",
            "n_projects",
        ]
    )
    .size()
    .unstack(fill_value=0)
)

Lineage breadth of recurrent associations:
        program        recurrent_scope  n_pairs  median_supported_projects  min_supported_projects  max_supported_projects  n_exactly_3_projects  n_at_least_5_projects  n_at_least_10_projects  fraction_exactly_3_projects  fraction_at_least_5_projects  fraction_at_least_10_projects
CONSENSUS_TX_01      program_exclusive     1203                        4.0                       3                      17                   350                    577                      49                     0.290939                      0.479634                       0.040732
CONSENSUS_TX_01 shared_across_programs      112                        8.0                       3                      17                    10                     93                      34                     0.089286                      0.830357                       0.303571
CONSENSUS_TX_02      program_exclusive       39                        5.0                       3             

In [67]:
# =============================================================================
# Relate lineage breadth to background-burden-conditioned effect stability
# =============================================================================

breadth_burden_relationship = (
    recurrent_lineage_breadth
    .merge(
        recurrent_review_table[
            [
                "Hugo_Symbol",
                "program",
                "burden_conditioned_same_direction",
            ]
        ],
        on=["Hugo_Symbol", "program"],
        how="left",
        validate="one_to_one",
    )
)

assert (
    breadth_burden_relationship[
        "burden_conditioned_relative_beta"
    ]
    .notna()
    .all()
)

assert (
    breadth_burden_relationship[
        "burden_conditioned_same_direction"
    ]
    .notna()
    .all()
)

breadth_burden_rows = []

for program, frame in breadth_burden_relationship.groupby(
    "program",
    sort=True,
):
    pearson_r, pearson_p = pearsonr(
        frame["n_projects"],
        frame["burden_conditioned_relative_beta"],
    )

    spearman_rho, spearman_p = spearmanr(
        frame["n_projects"],
        frame["burden_conditioned_relative_beta"],
    )

    breadth_burden_rows.append(
        {
            "program": program,
            "n_pairs": len(frame),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
        }
    )

breadth_burden_correlation = pd.DataFrame(
    breadth_burden_rows
)

print(
    "Correlation between lineage breadth and "
    "burden-conditioned relative effect:"
)
print(
    breadth_burden_correlation.to_string(index=False)
)

print("\nDirection retention by supported-project count:")
print(
    breadth_burden_relationship
    .groupby(
        [
            "program",
            "n_projects",
        ]
    )
    .agg(
        n_pairs=("Hugo_Symbol", "size"),
        fraction_direction_retained=(
            "burden_conditioned_same_direction",
            "mean",
        ),
        median_relative_beta=(
            "burden_conditioned_relative_beta",
            "median",
        ),
    )
    .to_string()
)

Correlation between lineage breadth and burden-conditioned relative effect:
        program  n_pairs  pearson_r    pearson_p  spearman_rho   spearman_p
CONSENSUS_TX_01     1315   0.142351 2.176616e-07      0.221101 5.032372e-16
CONSENSUS_TX_02       78   0.280773 1.277451e-02      0.439628 5.647399e-05
CONSENSUS_TX_03      208  -0.087094 2.109700e-01     -0.055524 4.257020e-01

Direction retention by supported-project count:
                            n_pairs fraction_direction_retained  median_relative_beta
program         n_projects                                                           
CONSENSUS_TX_01 3               360                    0.988889              0.440677
                4               285                    0.996491              0.496387
                5               209                         1.0              0.580300
                6               106                         1.0              0.596406
                7               114                    

In [68]:
# =============================================================================
# Build compact program-level statistical summary
# =============================================================================

program_summary_rows = []

for program in PROGRAM_COLUMNS:
    all_tests = final_gene_program_summary.loc[
        final_gene_program_summary["program"].eq(program)
    ]

    fdr_supported = all_tests.loc[
        all_tests["primary_fdr_supported"]
    ]

    recurrent = all_tests.loc[
        all_tests["cross_cancer_recurrent"]
    ]

    recurrent_partition_program = recurrent_partition.loc[
        recurrent_partition["program"].eq(program)
    ]

    program_summary_rows.append(
        {
            "program": program,
            "n_primary_tests": len(all_tests),
            "n_fdr_supported": len(fdr_supported),
            "n_cross_cancer_recurrent": len(recurrent),
            "n_unique_recurrent_genes": recurrent["Hugo_Symbol"].nunique(),
            "n_program_exclusive_recurrent_genes": int(
                recurrent_partition_program[
                    "recurrent_scope"
                ]
                .eq("program_exclusive")
                .sum()
            ),
            "n_sensitivity_stable_fdr_pairs": int(
                fdr_supported["sensitivity_stable"]
                .fillna(False)
                .sum()
            ),
            "median_primary_beta_recurrent": (
                recurrent["beta_mutation"].median()
            ),
            "median_supported_projects_recurrent": (
                recurrent["n_projects"].median()
            ),
            "median_directional_fraction_recurrent": (
                recurrent["directional_fraction"].median()
            ),
            "burden_conditioned_direction_retention_recurrent": (
                recurrent[
                    "burden_conditioned_same_direction"
                ].mean()
            ),
            "median_burden_conditioned_relative_beta_recurrent": (
                recurrent[
                    "burden_conditioned_relative_beta"
                ].median()
            ),
        }
    )

program_level_statistical_summary = pd.DataFrame(
    program_summary_rows
)

assert (
    program_level_statistical_summary[
        "n_primary_tests"
    ]
    .eq(len(primary_gene_universe))
    .all()
)

assert (
    program_level_statistical_summary[
        "n_cross_cancer_recurrent"
    ].sum()
    == 1601
)

print(
    program_level_statistical_summary.to_string(
        index=False
    )
)

        program  n_primary_tests  n_fdr_supported  n_cross_cancer_recurrent  n_unique_recurrent_genes  n_program_exclusive_recurrent_genes  n_sensitivity_stable_fdr_pairs  median_primary_beta_recurrent  median_supported_projects_recurrent  median_directional_fraction_recurrent  burden_conditioned_direction_retention_recurrent  median_burden_conditioned_relative_beta_recurrent
CONSENSUS_TX_01             4449             1665                      1315                      1315                                 1203                            1664                       0.275729                                  5.0                                  1.000                                          0.996198                                           0.506815
CONSENSUS_TX_02             4449               91                        78                        78                                   39                              90                      -0.135069                                  5.0    

In [72]:
# =============================================================================
# Persist downstream-consumable genomic-context characterization
# =============================================================================

OUTPUT_DIR = Paths.secondary_characterization
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_ASSOCIATIONS_OUTPUT_PATH = (
    OUTPUT_DIR
    / "450_primary_gene_program_associations.csv"
)

final_gene_program_summary.to_csv(
    PRIMARY_ASSOCIATIONS_OUTPUT_PATH,
    index=False,
)

reloaded_primary_associations = pd.read_csv(
    PRIMARY_ASSOCIATIONS_OUTPUT_PATH
)

assert reloaded_primary_associations.shape == (
    final_gene_program_summary.shape
)

assert reloaded_primary_associations[
    ["Hugo_Symbol", "program"]
].equals(
    final_gene_program_summary[
        ["Hugo_Symbol", "program"]
    ].reset_index(drop=True)
)

print(
    "Saved:",
    project_relative_path(
        PRIMARY_ASSOCIATIONS_OUTPUT_PATH
    ),
)
print(
    "Shape:",
    reloaded_primary_associations.shape,
)

Saved: data/processed/secondary_characterization/450_primary_gene_program_associations.csv
Shape: (13347, 30)


## Notebook closure

Notebook 450 completes the secondary somatic genomic-context characterization
of the three frozen Phase 4 consensus tumor programs.

Across the complete primary family of 13,347 gene × program tests, 1,978
associations met the global FDR threshold and 1,601 additionally satisfied the
prespecified lineage-aware `cross_cancer_recurrent` criteria.

Prespecified sensitivity analyses showed high directional stability among the
primary FDR-supported associations. However, the exploratory focal-excluded
background observed-variant-burden diagnostic showed substantial attenuation
for many associations, particularly for `CONSENSUS_TX_03`. Cross-cancer
recurrence therefore remains distinct from evidence of gene-specific genomic
context independent of broader observed mutational load.

Mutation-resource availability is lineage dependent and is not assumed to be
missing completely at random. Conventional TMB is not reported because a
defensible sample-comparable callable-territory denominator is unavailable.

The complete primary gene × program association family is persisted as:

`data/processed/secondary_characterization/450_primary_gene_program_associations.csv`

These results provide secondary genomic-context evidence for downstream
integration. They do not modify the frozen consensus programs or establish
causal mechanism, driver status, clinical relevance, or therapeutic validity.